# A100 ablation notebook -- Predictive AI Evaluation Challenge

Compares three models under item cold-start validation:
1. k-factor / neural-IRT
2. k-factor + ordinary MLP residual
3. k-factor + gated MLP / SwiGLU residual

Heavy lifting:
- Downloads `aims-foundations/measurement-db` from Hugging Face.
- Embeds every unique item / subject text once with a configurable HF
  encoder (default: `Qwen/Qwen3-Embedding-4B`) on the A100.
- Trains all three model variants across multiple seeds.
- Reports primary metric = item-cold-start val log-loss (lower is better).
- Lets you pick which run to export to a Codabench-compatible submission.

IMPORTANT
---------
Primary selection MUST happen on the item-cold-start split, NOT random-row.
Random-row validation is reported only as a sanity comparator; if a model
only improves random-row but not item-cold-start, the notebook flags it.

## 0. Clone the project repo (Colab / fresh Vertex AI instances only)

When running on Colab or a fresh Vertex AI Workbench instance, this cell
clones the project repo so that ``src/``, ``configs/`` and ``scripts/`` are
available. If those folders already exist next to the notebook (e.g. you
launched Jupyter from inside the repo), it does nothing.

To use a private fork, change ``REPO_URL`` and optionally set the
``GIT_AUTH_TOKEN`` env var (or paste a token into ``REPO_URL`` directly).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/bwathomas/prediction-competition-321M.git"
REPO_NAME = "prediction-competition-321M"
REPO_BRANCH = os.environ.get("REPO_BRANCH", "main")


def _bootstrap_repo() -> Path:
    """Return the absolute path to the repo root, cloning if necessary.

    Search order:
    1. parent of this notebook (when run from the repo directly).
    2. ``./{REPO_NAME}`` under the current working directory (Colab convention).
    3. clone fresh into the cwd.
    """
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(globals()["__file__"]).resolve().parent.parent)
    candidates.append(Path.cwd() / REPO_NAME)
    candidates.append(Path.cwd())

    for cand in candidates:
        if (cand / "src" / "data.py").is_file():
            print(f"[bootstrap] using existing repo at {cand}")
            return cand

    target = Path.cwd() / REPO_NAME
    if target.exists():
        print(f"[bootstrap] {target} exists but is incomplete; pulling latest")
        subprocess.run(
            ["git", "-C", str(target), "fetch", "--depth", "1", "origin", REPO_BRANCH],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(target), "reset", "--hard", f"origin/{REPO_BRANCH}"],
            check=True,
        )
    else:
        clone_url = REPO_URL
        token = os.environ.get("GIT_AUTH_TOKEN", "").strip()
        if token and clone_url.startswith("https://github.com/"):
            # Inject token without ever printing it.
            clone_url = clone_url.replace(
                "https://github.com/", f"https://{token}@github.com/"
            )
        print(f"[bootstrap] cloning into {target}")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(target)],
            check=True,
        )
    return target


ROOT = _bootstrap_repo()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print(f"ROOT (cwd)    : {ROOT}")

## 1. Install pinned requirements and import the stack

In [ ]:
import importlib
import re
import subprocess
import sys
import time

REQUIREMENTS_PATH = ROOT / "requirements.txt"
INSTALL_REQUIREMENTS = bool(int(os.environ.get("INSTALL_REQUIREMENTS", "1")))
if INSTALL_REQUIREMENTS and REQUIREMENTS_PATH.exists():
    print(f"[bootstrap] pip install -r {REQUIREMENTS_PATH}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS_PATH)],
        check=False,
    )


def _tail(proc: subprocess.CompletedProcess, n: int = 600) -> str:
    return (proc.stderr or proc.stdout or "")[-n:].strip()


def _verify_flash_attn_import() -> bool:
    """Return True only if flash-attn AND its compiled CUDA extension import.

    Importing only the Python package is not enough -- if the compiled
    ``flash_attn_2_cuda`` ABI doesn't match the running torch / cuda /
    glibc combo, ``from flash_attn import flash_attn_func`` silently
    succeeds while the first call from the encoder dies with a vague
    ``undefined symbol`` error. We import the extension directly here so
    that mismatch surfaces at install time instead of training time.
    """
    try:
        importlib.invalidate_caches()
        import flash_attn  # type: ignore  # noqa: F401
        import flash_attn_2_cuda  # type: ignore  # noqa: F401

        print(f"[flash-attn] import verified: {flash_attn.__version__}")
        return True
    except Exception as exc:  # noqa: BLE001
        print(f"[flash-attn] import verification failed: {exc}")
        return False


def _pip_install_wheel_url(wheel_url: str, label: str, timeout_s: int = 300) -> bool:
    """Install a direct wheel URL without dep-resolution or source build."""
    print(f"[flash-attn] trying {label}")
    print(f"[flash-attn] url   : {wheel_url}")
    t0 = time.time()
    try:
        proc = subprocess.run(
            [
                sys.executable, "-m", "pip", "install", "-q",
                "--force-reinstall", "--no-deps", wheel_url,
            ],
            capture_output=True,
            text=True,
            timeout=timeout_s,
        )
    except subprocess.TimeoutExpired:
        print(f"[flash-attn] {label} timed out after {timeout_s}s")
        return False
    if proc.returncode != 0:
        print(f"[flash-attn] {label} failed in {time.time() - t0:.1f}s")
        tail = _tail(proc)
        if tail:
            print(f"[flash-attn] last lines:\n{tail}")
        return False
    print(f"[flash-attn] {label} installed in {time.time() - t0:.1f}s")
    return _verify_flash_attn_import()


def _install_flash_attention() -> bool:
    """Install Flash Attention 2 from a prebuilt wheel matching this runtime.

    The PyPI ``flash-attn`` package defaults to building from source via
    nvcc, which takes 20-40 minutes on Colab and frequently OOMs. We avoid
    that entirely by enumerating candidate prebuilt wheel URLs derived
    from torch's CUDA version, the *runtime* C++ ABI flag, and the
    Python version, then pip-installing each in turn until one verifies.

    A wheel is considered installed only when both ``flash_attn`` and the
    compiled ``flash_attn_2_cuda`` extension import without raising. The
    encoder's SDPA path keeps the pipeline alive if no candidate verifies.
    """
    try:
        import torch as _torch  # torch was just pip-installed above
    except Exception:
        print("[flash-attn] torch unavailable; skipping install")
        return False
    if not _torch.cuda.is_available():
        print("[flash-attn] no CUDA device; skipping install (SDPA fallback)")
        return False

    if _verify_flash_attn_import():
        return True

    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_base = re.split(r"[+]", _torch.__version__)[0]
    torch_major_minor = ".".join(torch_base.split(".")[:2])
    cuda_version = _torch.version.cuda or ""
    cuda_major = cuda_version.split(".")[0] if cuda_version else "0"
    runtime_cxx11_abi = (
        "TRUE" if bool(getattr(_torch._C, "_GLIBCXX_USE_CXX11_ABI", False)) else "FALSE"
    )
    print(
        "[flash-attn] target: "
        f"py={py_tag} torch={torch_major_minor} cuda={cuda_version or 'unknown'} "
        f"cxx11abi={runtime_cxx11_abi}"
    )

    # Enumerate official Dao-AILab release wheels. Trying multiple versions /
    # ABIs is cheap -- pip returns instantly on a 404 -- and protects us from
    # a single missing asset blocking the whole install.
    candidates: list[tuple[str, str]] = []
    for fa_version in ("2.8.3", "2.7.4.post1"):
        for abi in (runtime_cxx11_abi, "FALSE", "TRUE"):
            wheel_name = (
                f"flash_attn-{fa_version}+cu{cuda_major}torch{torch_major_minor}"
                f"cxx11abi{abi}-{py_tag}-{py_tag}-linux_x86_64.whl"
            )
            wheel_url = (
                "https://github.com/Dao-AILab/flash-attention/releases/download/"
                f"v{fa_version}/{wheel_name}"
            )
            candidates.append(
                (f"official wheel: flash-attn {fa_version} abi={abi}", wheel_url)
            )

    seen: set[str] = set()
    for label, url in candidates:
        if url in seen:
            continue
        seen.add(url)
        if _pip_install_wheel_url(url, label):
            return True

    print(
        "[flash-attn] no compatible prebuilt wheel installed; using PyTorch SDPA fallback. "
        "Not running `pip install flash-attn` to avoid a 30 min source build."
    )
    return False


INSTALL_FLASH_ATTN = bool(int(os.environ.get("INSTALL_FLASH_ATTN", "1")))
if INSTALL_REQUIREMENTS and INSTALL_FLASH_ATTN:
    _install_flash_attention()


def _install_faiss_gpu() -> bool:
    """Upgrade ``faiss-cpu`` -> ``faiss-gpu-cu12`` when running on CUDA 12.

    The training-time k-means path in ``src.clustering`` automatically uses
    FAISS GPU when ``faiss.get_num_gpus() > 0``; that requires the
    ``faiss-gpu-cu12`` wheel (the CPU build reports zero GPUs). On
    non-GPU / non-CUDA-12 environments we leave the existing ``faiss-cpu``
    install alone and the clustering step transparently falls back to
    sklearn CPU.
    """
    try:
        import torch as _torch
    except Exception:
        print("[faiss-gpu] torch unavailable; keeping faiss-cpu")
        return False
    if not _torch.cuda.is_available():
        print("[faiss-gpu] no CUDA device; keeping faiss-cpu")
        return False
    cuda_major = _torch.version.cuda.split(".")[0] if _torch.version.cuda else "0"
    if cuda_major != "12":
        print(f"[faiss-gpu] CUDA {cuda_major}.x: only cu12 wheel is published; skipping")
        return False
    try:
        import faiss  # type: ignore

        if int(getattr(faiss, "get_num_gpus", lambda: 0)()) > 0:
            print("[faiss-gpu] already installed with GPU support")
            return True
    except Exception:
        pass
    # Uninstall the CPU build first so the GPU wheel doesn't end up
    # shadowed by stale faiss-cpu files on disk.
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-q", "-y", "faiss-cpu"],
        capture_output=True,
        text=True,
    )
    print("[faiss-gpu] installing faiss-gpu-cu12 ...")
    proc = subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--upgrade", "--no-cache-dir", "faiss-gpu-cu12",
        ],
        capture_output=True,
        text=True,
    )
    if proc.returncode != 0:
        tail = (proc.stderr or proc.stdout or "")[-400:].strip()
        print(f"[faiss-gpu] install failed; clustering will use sklearn CPU\n{tail}")
        return False
    # Drop any faiss modules already loaded as CPU. Otherwise the existing
    # process keeps using the CPU extension and the GPU wheel never takes
    # effect until kernel restart.
    for mod_name in [m for m in list(sys.modules) if m == "faiss" or m.startswith("faiss.")]:
        sys.modules.pop(mod_name, None)
    importlib.invalidate_caches()
    try:
        import faiss  # type: ignore  # noqa: F401

        n = int(faiss.get_num_gpus())
        print(f"[faiss-gpu] installed; visible GPUs: {n}")
        return n > 0
    except Exception as exc:  # noqa: BLE001
        print(f"[faiss-gpu] post-install import failed: {exc}")
        return False


INSTALL_FAISS_GPU = bool(int(os.environ.get("INSTALL_FAISS_GPU", "1")))
if INSTALL_REQUIREMENTS and INSTALL_FAISS_GPU:
    _install_faiss_gpu()


import json
import logging
from dataclasses import asdict

import numpy as np
import pandas as pd
import torch
import yaml

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(name)s %(levelname)s %(message)s",
    datefmt="%H:%M:%S",
)
LOG = logging.getLogger("notebook")

print(f"Python        : {sys.version.split()[0]}")
print(f"Torch         : {torch.__version__}")
print(f"CUDA          : {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")

## 2. Environment / GPU banner
Fails loudly if no CUDA device is present (set ``ALLOW_CPU=1`` to override).

In [ ]:
from src.embeddings import print_gpu_banner

print_gpu_banner(allow_cpu=bool(int(os.environ.get("ALLOW_CPU", "0"))))

## 3. Load configuration

Override anything you like by editing the ``CFG`` dict before running the
downstream cells. The encoder model id is the most common knob.

In [ ]:
with open(ROOT / "configs" / "default.yaml", "r", encoding="utf-8") as fh:
    CFG = yaml.safe_load(fh)

# Quick overrides for fast iteration. Tweak these in-place.
# CFG["data"]["max_rows_per_benchmark"] = 5000
# CFG["encoder"]["model_id"] = "sentence-transformers/all-mpnet-base-v2"
# CFG["train"]["epochs"] = 5
# CFG["train"]["seeds"] = [0]

print(json.dumps(CFG, indent=2))

## 4. Resolve HF_TOKEN and log in to Hugging Face

Order:
1. ``HF_TOKEN`` environment variable
2. Google Colab ``userdata.get('HF_TOKEN')`` secret (auto on Colab; create
   it once via the Secrets panel and grant this notebook access)
3. Google Secret Manager secret named ``HF_TOKEN`` (if running on GCP and
   google-cloud-secret-manager is installed)
4. Interactive ``getpass`` prompt

The token is **never** logged or written to disk.

In [ ]:
from src.embeddings import login_huggingface, resolve_hf_token

HF_TOKEN = resolve_hf_token(interactive=True)
login_huggingface(HF_TOKEN)

## 5. Download + load + key the dataset

Downloads the per-benchmark parquet files into ``artifacts/data/`` (idempotent),
joins them with the registry tables, normalizes ``condition``, builds
stable ``subject_key`` / ``item_key`` / ``benchmark_condition_key`` columns,
and reports descriptive statistics.

In [ ]:
from src.data import (
    DatasetStats,
    compute_dataset_stats,
    prepare_dataset,
    print_dataset_stats,
)

df = prepare_dataset(CFG["data"], token=HF_TOKEN, download=True)
print(f"Final dataset rows: {len(df):,}")
print(df.head(3).to_dict(orient="records"))

stats = compute_dataset_stats(df)
print_dataset_stats(stats)

## 6. Build the validation splits

- **item_cold_start** (PRIMARY): val item_keys disjoint from train item_keys.
- **benchmark_heldout** (optional): hold out whole benchmarks.
- **random_row_debug** (LEAKY): shuffle-by-row. ONLY for sanity comparison.

In [ ]:
from src.data import (
    make_benchmark_heldout_split,
    make_item_cold_start_split,
    make_random_row_split,
)
from src.sanity_checks import (
    print_results,
    run_data_checks,
    to_dataframe,
)

SEED = int(CFG["seed"])
split_cfg = CFG["splits"]

splits = {}

splits["item_cold_start"] = make_item_cold_start_split(
    df,
    val_fraction=float(split_cfg["val_fraction"]),
    seed=SEED,
    holdout_benchmarks=split_cfg.get("holdout_benchmarks") or None,
)

if split_cfg.get("holdout_benchmarks"):
    splits["benchmark_heldout"] = make_benchmark_heldout_split(
        df,
        holdout_benchmarks=split_cfg["holdout_benchmarks"],
        seed=SEED,
    )

if split_cfg.get("enable_random_row_debug", False):
    splits["random_row_debug"] = make_random_row_split(
        df, val_fraction=float(split_cfg["val_fraction"]), seed=SEED
    )

for name, art in splits.items():
    print(
        f"[{name}] train={len(art.train):>9,}  val={len(art.val):>7,}  "
        f"val_unseen_subject={len(art.val_unseen_subject):>5,}  notes={art.notes}"
    )

## 7. Data sanity checks

Required columns, label range, leakage, key stability, duplicate /
inconsistent rows.

In [ ]:
data_checks = run_data_checks(
    df,
    train=splits["item_cold_start"].train,
    val=splits["item_cold_start"].val,
)
print_results(data_checks)

## 8. Build the encoder and embed unique items / subjects

We embed each unique ``item_key`` and ``subject_key`` once and persist the
result to ``artifacts/embeddings/{encoder_slug}/`` as ``items.parquet`` /
``subjects.parquet`` + ``meta.json`` + ``encoding_log.json``.

When ``drive_cache.enabled`` is true (the Colab default) we first try to
pull a previously-encoded cache from Google Drive. On a content-hash hit
the encoder is never even loaded.

In [ ]:
import time
import warnings

from tqdm.auto import tqdm

from src.embeddings import (
    EncoderConfig,
    TransformerEmbedder,
    assert_deduplicated,
    build_unique_items,
    build_unique_subjects,
    content_hash_for_items,
    encoder_slug as _encoder_slug,
    verify_flash_attention,
)
from src import drive_cache as drive_cache_mod


def _fmt_time(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m}m {s}s"
    if m:
        return f"{m}m {s}s"
    return f"{s}s"


# Verify Flash Attention 2 is actually usable before constructing the
# embedder. The Python `flash_attn` shim can import even when the CUDA
# kernel binding is missing or ABI-mismatched against the running torch;
# verifying both eagerly here lets us downgrade to SDPA in the config
# instead of crashing at the first forward pass.
requested_fa = bool(CFG["encoder"].get("use_flash_attention", False))
fa_active, fa_msg = verify_flash_attention(requested_fa)
print(f"Flash Attention 2   : {'ACTIVE' if fa_active else 'OFF'} -- {fa_msg}")
if requested_fa and not fa_active:
    warnings.warn(
        "use_flash_attention=True in config but flash_attn is not importable. "
        "Downgrading to SDPA for this run."
    )
    CFG["encoder"]["use_flash_attention"] = False

# Encoder defaults are loaded from configs/default.yaml. Override CFG["encoder"]
# here if you want to A/B different encoders without editing the yaml.
enc_cfg = EncoderConfig(**CFG["encoder"])
embedder = TransformerEmbedder(enc_cfg)
slug = _encoder_slug(enc_cfg.model_id)

print(f"Encoder             : {enc_cfg.model_id}")
print(f"Embedding cache dir : {embedder.base}")
print(f"Batch size (config) : {enc_cfg.batch_size} (fallback {enc_cfg.batch_size_fallback})")
print(f"max_length          : {enc_cfg.max_length or 'auto (99th pct, /64)'}")
print(f"Use Flash Attn 2    : {enc_cfg.use_flash_attention}")
print(f"Pooling             : {enc_cfg.pooling}")
print(f"Contextual items    : {enc_cfg.use_contextual_item_text}")

# Build the dedup'd item / subject lists. Dedup is verified loudly at the
# top of every encode call -- duplicate forwards are typically the largest
# source of wasted encoder time.
required_cols = {"item_key", "benchmark", "condition", "item_content"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"df is missing required item columns: {sorted(missing)}")
required_cols = {"subject_key", "subject_content"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"df is missing required subject columns: {sorted(missing)}")

item_df = (
    df[["item_key", "benchmark", "condition", "item_content"]]
    .drop_duplicates(subset=["item_key"])
    .reset_index(drop=True)
)
item_keys_list, item_texts_list, item_benches_list = build_unique_items(
    item_df,
    contextual=enc_cfg.use_contextual_item_text,
    passage_prefix=embedder._resolve_passage_prefix(),
)
assert_deduplicated(item_keys_list, kind="item")

subject_df = (
    df[["subject_key", "subject_content"]]
    .drop_duplicates(subset=["subject_key"])
    .reset_index(drop=True)
)
subject_keys_list, subject_texts_list = build_unique_subjects(
    subject_df, query_prefix=embedder._resolve_query_prefix()
)
assert_deduplicated(subject_keys_list, kind="subject")

# Content hash detects whether the underlying texts changed since the last
# encoder run; persisted to meta.json so future runs (including the Drive
# cache) can do a strict "skip encoding" check.
item_pairs = list(zip(item_keys_list, item_texts_list))
subject_pairs = list(zip(subject_keys_list, subject_texts_list))
CONTENT_HASH = content_hash_for_items(item_pairs + subject_pairs)
print(f"Content hash        : {CONTENT_HASH[:16]}...  (over {len(item_pairs):,} items + {len(subject_pairs):,} subjects)")

# Phase-level progress bar so a long encoding pass shows where we are at a
# glance. The per-batch progress comes from the embedder's built-in tqdm.
phases = [
    "drive-resolve",
    "warm-cache",
    "encode-items",
    "encode-subjects",
    "finalize",
    "drive-upload",
]
phase_pbar = tqdm(phases, desc="Embedding pipeline", leave=True)
phase_times: dict[str, float] = {}


def _next_phase(name: str) -> float:
    phase_pbar.set_postfix_str(name)
    phase_pbar.update(1)
    return time.time()


# 1. Drive cache lookup (no-op outside Colab / when disabled).
t0 = _next_phase("drive-resolve")
cache_root = ROOT / CFG["encoder"]["cache_dir"]
drive_status = drive_cache_mod.resolve_cache(
    cfg=CFG,
    encoder_slug=slug,
    local_cache_root=cache_root,
    expected_hash=CONTENT_HASH,
)
phase_times["drive_resolve"] = float(time.time() - t0)
print(f"\nDrive cache decision: {drive_status.reason}")
print(json.dumps(drive_status.as_dict(), indent=2))

# 2. Warm in-memory caches from disk (drive sync writes parquet files there).
t0 = _next_phase("warm-cache")
embedder.warm_caches_from_disk()
phase_times["warm_cache"] = float(time.time() - t0)

# 3. Encode whatever is still missing. On a strict drive hit every key is
#    already in the parquet cache and the encoder is never loaded.
t0 = _next_phase("encode-items")
print(f"\nEncoding unique items: {len(item_keys_list):,}")
item_emb_lookup, item_log = embedder.embed_unique(
    kind="item",
    keys=item_keys_list,
    texts=item_texts_list,
    benchmarks=item_benches_list,
)
phase_times["items"] = float(time.time() - t0)
LOG.info(
    "items: total=%d cached=%d encoded=%d elapsed=%s",
    item_log["n_total"],
    item_log["n_cache_hits"],
    item_log["n_encoded"],
    _fmt_time(phase_times["items"]),
)

t0 = _next_phase("encode-subjects")
print(f"\nEncoding unique subjects: {len(subject_keys_list):,}")
subject_emb_lookup, subject_log = embedder.embed_unique(
    kind="subject",
    keys=subject_keys_list,
    texts=subject_texts_list,
)
phase_times["subjects"] = float(time.time() - t0)
LOG.info(
    "subjects: total=%d cached=%d encoded=%d elapsed=%s",
    subject_log["n_total"],
    subject_log["n_cache_hits"],
    subject_log["n_encoded"],
    _fmt_time(phase_times["subjects"]),
)

# 4. Persist parquet caches + meta + encoding_log.json.
t0 = _next_phase("finalize")
embedder.finalize(
    content_hash=CONTENT_HASH,
    n_items=len(item_keys_list),
    n_subjects=len(subject_keys_list),
    extra_log={
        "items": item_log,
        "subjects": subject_log,
        "drive_cache": drive_status.as_dict(),
        "phase_seconds": phase_times,
        "flash_attention_active": fa_active,
    },
)
phase_times["finalize"] = float(time.time() - t0)

# 5. Upload to Drive if anything new was encoded (or if the on-disk hash
#    differed from Drive's).
t0 = _next_phase("drive-upload")
drive_cfg = (CFG.get("drive_cache") or {})
if (
    drive_cfg.get("enabled")
    and drive_cfg.get("upload_on_completion", True)
    and drive_status.mounted
):
    if (
        drive_status.cache_hit
        and item_log["n_encoded"] == 0
        and subject_log["n_encoded"] == 0
    ):
        print("Drive cache up to date; skipping upload.")
    else:
        drive_folder = Path(drive_cfg["folder"]) / slug
        upload_summary = drive_cache_mod.upload_from_local(
            local_folder=embedder.base,
            drive_folder=drive_folder,
        )
        print(f"Drive upload: {json.dumps(upload_summary, indent=2)}")
elif drive_cfg.get("enabled") and not drive_status.mounted:
    print("Drive cache enabled but mount unavailable -- skipping upload.")
phase_times["drive_upload"] = float(time.time() - t0)
phase_pbar.close()

emb_stats = embedder.stats.report()
print("\nPhase timings:")
for k, v in phase_times.items():
    print(f"  {k:<16s} {_fmt_time(v)}")
print("\nEncoder diagnostics:")
print(json.dumps(emb_stats, indent=2))

## 8b. Pool features and k-means clustering on items

Two cheap item-side channels that the new Item-IRT variants compose with
the embedding:

- **Pool features**: hand-engineered scalars (token / char length, has-latex,
  has-code, question count, MC indicators, language). Computed once per
  unique item, cached to ``artifacts/item_features/pool_features.parquet``,
  z-scored using *train-only* stats persisted alongside.
- **Clusters**: k-means on the cached item embeddings. Centroids go to
  ``artifacts/cluster_centroids.npy`` and per-item assignments to
  ``artifacts/item_clusters.parquet``. Cluster id 0 is reserved for UNK.

In [ ]:
from src.clustering import fit_and_assign, load_centroids
from src.item_features import (
    POOL_FEATURE_NAMES,
    apply_zscore,
    compute_features_for_items,
    fit_zscore_stats,
    load_pool_features,
    load_zscore_stats,
    save_pool_features,
    save_zscore_stats,
)

def try_load_cached_clusters(
    *,
    centroids_path,
    assignments_path,
    k: int,
    d: int,
    item_keys,
):
    """
    Local replacement for missing src.clustering.try_load_cached_clusters.

    Returns:
        (centroids, cluster_assignments_dict) on valid cache hit
        None on cache miss / invalid cache
    """
    import numpy as np
    import pandas as pd
    from pathlib import Path

    centroids_path = Path(centroids_path)
    assignments_path = Path(assignments_path)
    item_keys = [str(x) for x in item_keys]

    if not centroids_path.exists() or not assignments_path.exists():
        print("[clusters] cache miss: centroids or assignments file missing")
        return None

    try:
        centroids = np.load(centroids_path)
    except Exception as exc:
        print(f"[clusters] cache miss: failed to load centroids: {exc}")
        return None

    if centroids.shape != (int(k), int(d)):
        print(
            f"[clusters] cache miss: centroid shape {centroids.shape} "
            f"!= expected {(int(k), int(d))}"
        )
        return None

    try:
        assign_df = pd.read_parquet(assignments_path)
    except Exception as exc:
        print(f"[clusters] cache miss: failed to read assignments parquet: {exc}")
        return None

    if "item_key" not in assign_df.columns:
        print(f"[clusters] cache miss: assignments missing item_key column; columns={list(assign_df.columns)}")
        return None

    cluster_col = None
    for c in ["cluster_id", "cluster", "cluster_idx", "assignment"]:
        if c in assign_df.columns:
            cluster_col = c
            break

    if cluster_col is None:
        print(f"[clusters] cache miss: no cluster column found; columns={list(assign_df.columns)}")
        return None

    if len(assign_df) != len(item_keys):
        print(
            f"[clusters] cache miss: assignments rows={len(assign_df):,} "
            f"!= expected item_keys={len(item_keys):,}"
        )
        return None

    cached_keys = assign_df["item_key"].astype(str).tolist()
    if set(cached_keys) != set(item_keys):
        missing = list(set(item_keys).difference(cached_keys))[:5]
        extra = list(set(cached_keys).difference(item_keys))[:5]
        print(
            "[clusters] cache miss: assignment item_key set mismatch; "
            f"missing examples={missing}, extra examples={extra}"
        )
        return None

    cluster_assignments = dict(
        zip(assign_df["item_key"].astype(str), assign_df[cluster_col].astype(int))
    )

    vals = np.fromiter(cluster_assignments.values(), dtype=np.int64)
    if vals.size:
        print(
            f"[clusters] cache hit: {len(cluster_assignments):,} assignments, "
            f"cluster id range=[{vals.min()}, {vals.max()}], "
            f"centroids shape={centroids.shape}"
        )
    else:
        print("[clusters] cache miss: empty assignments")
        return None

    return centroids.astype(np.float32, copy=False), cluster_assignments

ITEM_FEATURES_CFG = CFG.get("item_features", {}) or {}
USE_POOL_FEATURES = bool(ITEM_FEATURES_CFG.get("use_pool", True))
USE_CLUSTER_FEATURES = bool(ITEM_FEATURES_CFG.get("use_clusters", True))
POOL_FEATURE_DIM = int(ITEM_FEATURES_CFG.get("pool_feature_dim", len(POOL_FEATURE_NAMES)))
CLUSTER_EMBED_DIM = int(ITEM_FEATURES_CFG.get("cluster_embed_dim", 16))
# Guard against the contradictory `use_clusters=True, cluster_embed_dim=0`
# state -- which would otherwise propagate into ModelConfig and silently
# disable the cluster channel (has_cluster_embedding requires dim > 0)
# while still printing `use_cluster_features: True, n_clusters: 64` in the
# run banner. Coerce to the configs/default.yaml default so the channel
# actually trains.
if USE_CLUSTER_FEATURES and CLUSTER_EMBED_DIM <= 0:
    print(
        f"WARN: item_features.cluster_embed_dim={CLUSTER_EMBED_DIM} but "
        "use_clusters=True; coercing CLUSTER_EMBED_DIM=16 so the cluster "
        "channel is actually live."
    )
    CLUSTER_EMBED_DIM = 16
POOL_FEATURES_DIR = ROOT / ITEM_FEATURES_CFG.get("cache_dir", "artifacts/item_features")
POOL_FEATURES_PATH = POOL_FEATURES_DIR / "pool_features.parquet"
POOL_STATS_PATH = POOL_FEATURES_DIR / "pool_features_stats.json"

CLUSTERING_CFG = CFG.get("clustering", {}) or {}
N_CLUSTERS = int(CLUSTERING_CFG.get("k", 64))
CLUSTERING_SEED = int(CLUSTERING_CFG.get("seed", 0))
CENTROIDS_PATH = ROOT / CLUSTERING_CFG.get(
    "centroids_path", "artifacts/cluster_centroids.npy"
)
ASSIGNMENTS_PATH = ROOT / CLUSTERING_CFG.get(
    "assignments_path", "artifacts/item_clusters.parquet"
)
# FAISS GPU k-means controls. For closest-to-old sklearn behavior, use
# niter=100, nredo=4. For faster feature engineering, niter=30-50 and
# nredo=1 is usually enough on A100s.
FAISS_NITER = int(CLUSTERING_CFG.get("faiss_niter", 50))
FAISS_NREDO = int(CLUSTERING_CFG.get("faiss_nredo", 1))
FAISS_ASSIGN_BATCH = int(CLUSTERING_CFG.get("faiss_assign_batch", 65536))
FAISS_GPU_ID = int(CLUSTERING_CFG.get("gpu_id", 0))
CLUSTERING_BACKEND = str(CLUSTERING_CFG.get("backend", "auto"))
OVERWRITE_CLUSTERS = bool(CLUSTERING_CFG.get("overwrite", False))
# null/None preserves FAISS default (256 points/centroid). Raise to ~1024
# on large corpora when the FAISS subsample noticeably hurts centroid quality.
FAISS_MAX_POINTS_PER_CENTROID = CLUSTERING_CFG.get("faiss_max_points_per_centroid")
if FAISS_MAX_POINTS_PER_CENTROID is not None:
    FAISS_MAX_POINTS_PER_CENTROID = int(FAISS_MAX_POINTS_PER_CENTROID)

# 1) Pool features ----------------------------------------------------------
if USE_POOL_FEATURES:
    pool_df = load_pool_features(POOL_FEATURES_PATH)
    if pool_df is None or set(POOL_FEATURE_NAMES).difference(pool_df.columns):
        print(f"Computing pool features for {len(item_df):,} unique items ...")
        pool_df = compute_features_for_items(item_df, progress=True)
        save_pool_features(pool_df, POOL_FEATURES_PATH)
        print(f"Cached pool features -> {POOL_FEATURES_PATH.relative_to(ROOT)}")
    else:
        print(f"Loaded cached pool features ({len(pool_df):,} rows) from {POOL_FEATURES_PATH.relative_to(ROOT)}")
else:
    pool_df = None
    print("Pool features disabled (CFG.item_features.use_pool = false).")

print(f"Pool feature columns: {list(POOL_FEATURE_NAMES)}")

# 2) k-means on item embeddings -------------------------------------------
# Uses FAISS GPU k-means when faiss-gpu is available (~50-100x faster than
# sklearn on full-corpus A100 runs); falls back to sklearn CPU otherwise.
# Both paths produce identical on-disk artifacts.
if USE_CLUSTER_FEATURES:
    item_keys_for_clusters = item_df["item_key"].astype(str).tolist()
    emb_dim = int(embedder.embedding_dim)

    # Cache check FIRST. On a hit we skip not just the FAISS fit but also
    # the O(N) item-key membership validation, the O(N*D) matrix allocation,
    # and the per-item Python loop that copies vectors out of the lookup
    # dict. None of that is needed when centroids + assignments are already
    # on disk for this exact key set.
    cached = (
        None
        if OVERWRITE_CLUSTERS
        else try_load_cached_clusters(
            centroids_path=CENTROIDS_PATH,
            assignments_path=ASSIGNMENTS_PATH,
            k=N_CLUSTERS,
            d=emb_dim,
            item_keys=item_keys_for_clusters,
        )
    )
    if cached is not None:
        centroids, cluster_assignments = cached
        print(
            f"Clusters: cache hit (k={N_CLUSTERS}, n_items={len(cluster_assignments):,}); "
            f"skipping item-key validation + matrix build + FAISS k-means"
        )
    else:
        from tqdm.auto import tqdm as _tqdm

        # Cache miss: validate that every key resolves in the encoder lookup
        # *before* allocating the (potentially multi-GB) matrix, then build
        # it and refit.
        missing_keys = [k for k in item_keys_for_clusters if k not in item_emb_lookup]
        if missing_keys:
            raise KeyError(
                f"Missing {len(missing_keys):,} item embeddings before clustering. "
                f"First missing keys: {missing_keys[:10]}"
            )
        item_emb_matrix = np.empty((len(item_keys_for_clusters), emb_dim), dtype=np.float32)
        for i, k in _tqdm(
            enumerate(item_keys_for_clusters),
            total=len(item_keys_for_clusters),
            desc="Building item embedding matrix",
            unit="item",
            leave=False,
        ):
            vec = np.asarray(item_emb_lookup[k], dtype=np.float32)
            if vec.shape != (emb_dim,):
                raise ValueError(
                    f"Embedding for item_key={k!r} has shape {vec.shape}; "
                    f"expected {(emb_dim,)}"
                )
            item_emb_matrix[i] = vec

        centroids, cluster_assignments = fit_and_assign(
            item_keys_for_clusters,
            item_emb_matrix,
            k=N_CLUSTERS,
            seed=CLUSTERING_SEED,
            centroids_path=CENTROIDS_PATH,
            assignments_path=ASSIGNMENTS_PATH,
            overwrite=OVERWRITE_CLUSTERS,
            niter=FAISS_NITER,
            nredo=FAISS_NREDO,
            gpu_id=FAISS_GPU_ID,
            assign_batch_size=FAISS_ASSIGN_BATCH,
            backend=CLUSTERING_BACKEND,
            faiss_max_points_per_centroid=FAISS_MAX_POINTS_PER_CENTROID,
        )
        # Drop the matrix promptly: it can easily be >1 GB on a Qwen
        # corpus and is not used past this point.
        del item_emb_matrix

    print(
        f"Clusters: k={N_CLUSTERS} centroids={CENTROIDS_PATH.relative_to(ROOT)} "
        f"assignments={ASSIGNMENTS_PATH.relative_to(ROOT)}"
    )
else:
    centroids = None
    cluster_assignments = None
    print("Cluster features disabled (CFG.item_features.use_clusters = false).")

## 8c. Score training items with LLM-as-judge (cached)

Run a local instruction-tuned LM (default ``Qwen/Qwen3-4B-Instruct-2507``)
as a judge over every unique ``(subject_key, item_key)`` pair that appears
in training data. For each pair we read the next-token distribution at
the "Answer:" position and extract four scalar features:

    [lp_yes, lp_no, lp_yes - lp_no, p_yes_renormalized]

These features are stored to ``artifacts/judge/{judge_slug}/scores.parquet``
keyed by ``(subject_key, item_key)`` and (when Colab Drive is available)
pushed to Google Drive so re-running the notebook skips the GPU pass.

The four features are fed *into* the residual MLP alongside the item
embedding, pool features, and cluster embedding -- the head learns where
to trust the judge rather than applying a global blend weight. See
``src/judge.py`` for the implementation and locked prompt template.

Two practical warnings:
  * The prompt template is locked once the cache is populated; changing
    it costs GPU-hours to re-score. The on-disk slug encodes the prompt
    version hash so accidental edits show up as a cache miss.
  * If ``CFG['judge']['enabled']`` is False this cell is a no-op and the
    downstream model is trained without judge features (regression test).

In [ ]:
# ---------------------------------------------------------------------
# Assemble judge cache from exact 4 shard folders:
#   shard_000_of_004
#   shard_001_of_004
#   shard_002_of_004
#   shard_003_of_004
#
# Ignore:
#   shard_000_of_003
# ---------------------------------------------------------------------

import os
import json
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd

from src.judge import (
    JUDGE_FEATURE_DIM,
    JudgeConfig,
    build_judge_matrix,
    compute_prompt_version,
    features_lookup_from_dataframe,
    judge_slug as _judge_slug_fn,
    load_cache as _load_judge_cache,
    write_meta_snapshot as _write_judge_meta,
)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"[judge-assemble] Drive mount skipped/failed: {exc}", flush=True)


def _mark(msg):
    print(f"[judge-assemble {time.strftime('%H:%M:%S')}] {msg}", flush=True)


def _fmt_bytes(n):
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"


def _dir_size(path):
    path = Path(path)
    if not path.exists():
        return 0
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return total


def _read_all_parquets(folder: Path, shard_id: int) -> pd.DataFrame:
    folder = Path(folder)
    files = sorted(folder.rglob("*.parquet"))

    print(f"\n[shard {shard_id}] folder: {folder}", flush=True)
    print(f"[shard {shard_id}] parquet files: {len(files)}", flush=True)

    if not files:
        raise FileNotFoundError(f"No parquet files found under {folder}")

    frames = []
    for p in files:
        try:
            df = pd.read_parquet(p)
            print(f"  {p.relative_to(folder)} rows={len(df):,} cols={len(df.columns)}", flush=True)
            if len(df):
                df["_source_shard"] = shard_id
                df["_source_parquet"] = str(p)
                frames.append(df)
        except Exception as exc:
            print(f"  FAILED {p}: {repr(exc)}", flush=True)

    if not frames:
        raise RuntimeError(f"No readable nonempty parquet files under {folder}")

    return pd.concat(frames, ignore_index=True)


def _pair_set(df: pd.DataFrame) -> set[tuple[str, str]]:
    if df is None or df.empty:
        return set()
    if "subject_key" not in df.columns or "item_key" not in df.columns:
        return set()
    return set(zip(df["subject_key"].astype(str), df["item_key"].astype(str)))


def _filter_to_input_pairs(cache_df: pd.DataFrame, judge_input_df: pd.DataFrame) -> pd.DataFrame:
    wanted = judge_input_df[["subject_key", "item_key"]].astype(str).drop_duplicates()

    out = cache_df.copy()
    out["subject_key"] = out["subject_key"].astype(str)
    out["item_key"] = out["item_key"].astype(str)

    out = (
        out.merge(wanted, on=["subject_key", "item_key"], how="inner")
        .drop_duplicates(subset=["subject_key", "item_key"], keep="last")
        .reset_index(drop=True)
    )
    return out


def _load_cache_safe(cfg_jdg):
    try:
        df = _load_judge_cache(cfg_jdg)
        return pd.DataFrame() if df is None else df
    except Exception as exc:
        print(f"[judge-assemble] _load_judge_cache failed: {repr(exc)}", flush=True)
        return pd.DataFrame()


# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

JUDGE_CFG = CFG.get("judge") or {}
JUDGE_ENABLED = bool(JUDGE_CFG.get("enabled", False))
JUDGE_FEATURES_IN_RESIDUAL = bool(JUDGE_CFG.get("feature_in_residual", True))

judge_features_lookup = {}
judge_dataframe = None
judge_pairs_cached_before = 0
judge_pairs_newly_scored = 0
judge_total_pairs = 0
judge_wall_time = 0.0
judge_slug = ""
judge_prompt_version = ""

if not JUDGE_ENABLED:
    print("Judge disabled; skipping.", flush=True)

else:
    cfg_jdg = JudgeConfig(
        model_id=str(JUDGE_CFG.get("model_id", "Qwen/Qwen3-4B-Instruct-2507")),
        batch_size=int(JUDGE_CFG.get("batch_size", 32)),
        max_prompt_tokens=int(JUDGE_CFG.get("max_prompt_tokens", 1024)),
        bf16=bool(JUDGE_CFG.get("bf16", True)),
        use_flash_attention=bool(JUDGE_CFG.get("use_flash_attention", True)),
        trust_remote_code=bool(JUDGE_CFG.get("trust_remote_code", False)),
        cache_dir=str(JUDGE_CFG.get("cache_dir", "artifacts/judge")),
        num_workers=int(JUDGE_CFG.get("num_workers", 0)),
        prefetch_batches=int(JUDGE_CFG.get("prefetch_batches", 2)),
        length_bucket=bool(JUDGE_CFG.get("length_bucket", True)),
        pin_memory=bool(JUDGE_CFG.get("pin_memory", True)),
    )

    judge_slug = _judge_slug_fn(cfg_jdg)
    judge_prompt_version = compute_prompt_version(cfg_jdg)

    jdc_cfg = CFG.get("judge_drive_cache") or {}
    if not jdc_cfg.get("folder"):
        raise ValueError("Missing CFG['judge_drive_cache']['folder'].")

    drive_judge_root = Path(jdc_cfg["folder"])
    shard_dirs = [
        drive_judge_root / "shard_000_of_004",
        drive_judge_root / "shard_001_of_004",
        drive_judge_root / "shard_002_of_004",
        drive_judge_root / "shard_003_of_004",
    ]

    local_cache_root = ROOT / cfg_jdg.cache_dir
    local_slug_dir = local_cache_root / judge_slug
    final_drive_slug_dir = drive_judge_root / judge_slug

    print("Judge model          :", cfg_jdg.model_id)
    print("Judge slug           :", judge_slug)
    print("Prompt version hash  :", judge_prompt_version)
    print("Drive judge root     :", drive_judge_root)
    print("Local assembled cache:", local_slug_dir)
    print("Final Drive cache    :", final_drive_slug_dir)

    print("\nShard dirs:")
    for p in shard_dirs:
        print(f"  {p} exists={p.exists()} size={_fmt_bytes(_dir_size(p))}")

    missing = [str(p) for p in shard_dirs if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing expected shard dirs:\n" + "\n".join(missing))

    os.chdir(ROOT)
    local_cache_root.mkdir(parents=True, exist_ok=True)

    # -----------------------------------------------------------------
    # Build judge input pairs
    # -----------------------------------------------------------------

    train_df = splits["item_cold_start"].train

    pair_cols = [
        "subject_key",
        "item_key",
        "benchmark",
        "condition",
        "subject_content",
        "item_content",
    ]
    missing_cols = [c for c in pair_cols if c not in train_df.columns]
    if missing_cols:
        raise ValueError(f"Training frame missing judge columns: {missing_cols}")

    judge_input_df = (
        train_df[pair_cols]
        .drop_duplicates(subset=["subject_key", "item_key"])
        .reset_index(drop=True)
    )
    judge_input_df["subject_key"] = judge_input_df["subject_key"].astype(str)
    judge_input_df["item_key"] = judge_input_df["item_key"].astype(str)

    input_keys = set(zip(judge_input_df["subject_key"], judge_input_df["item_key"]))
    judge_total_pairs = len(judge_input_df)

    print(f"\nJudge unique train pairs: {judge_total_pairs:,}")

    # -----------------------------------------------------------------
    # Read + concatenate the four shards
    # -----------------------------------------------------------------

    frames = []
    for sid, folder in enumerate(shard_dirs):
        df = _read_all_parquets(folder, sid)

        if "subject_key" not in df.columns or "item_key" not in df.columns:
            raise ValueError(
                f"Shard {sid} missing subject_key/item_key. "
                f"columns={list(df.columns)}"
            )

        df["subject_key"] = df["subject_key"].astype(str)
        df["item_key"] = df["item_key"].astype(str)

        shard_hits = len(input_keys.intersection(_pair_set(df)))
        print(
            f"[shard {sid}] total rows={len(df):,} "
            f"unique pairs={len(_pair_set(df)):,} "
            f"input hits={shard_hits:,}/{judge_total_pairs:,}",
            flush=True,
        )

        frames.append(df)

    assembled_df = pd.concat(frames, ignore_index=True)

    print("\nAssembled raw rows:", f"{len(assembled_df):,}")
    print("Assembled raw unique pairs:", f"{len(_pair_set(assembled_df)):,}")

    assembled_df = (
        assembled_df
        .drop_duplicates(subset=["subject_key", "item_key"], keep="last")
        .reset_index(drop=True)
    )

    assembled_hits = len(input_keys.intersection(_pair_set(assembled_df)))

    print("Assembled deduped rows:", f"{len(assembled_df):,}")
    print(f"Assembled input hits : {assembled_hits:,}/{judge_total_pairs:,}")

    judge_dataframe = _filter_to_input_pairs(assembled_df, judge_input_df)

    print("Filtered train-pair rows:", f"{len(judge_dataframe):,}")

    if len(judge_dataframe) == 0:
        raise RuntimeError("Assembled shards produced zero matching judge rows.")

    # -----------------------------------------------------------------
    # Write final assembled local cache
    # -----------------------------------------------------------------

    if local_slug_dir.exists():
        backup = local_slug_dir.with_name(local_slug_dir.name + f".bak_{int(time.time())}")
        shutil.move(str(local_slug_dir), str(backup))
        print(f"Moved old local cache to: {backup}")

    local_slug_dir.mkdir(parents=True, exist_ok=True)

    # Write multiple names so load_cache has a high chance of finding it
    # regardless of exact implementation.
    out_paths = [
        local_slug_dir / "cache.parquet",
        local_slug_dir / "judge_cache.parquet",
    ]

    for out_path in out_paths:
        judge_dataframe.to_parquet(out_path, index=False)
        print(
            f"Wrote {out_path} rows={len(judge_dataframe):,} "
            f"size={_fmt_bytes(out_path.stat().st_size)}"
        )

    _write_judge_meta(cfg_jdg, n_rows=int(len(judge_dataframe)))

    # -----------------------------------------------------------------
    # Verify. If src.judge.load_cache doesn't pick up our written filename,
    # still use judge_dataframe directly.
    # -----------------------------------------------------------------

    loaded_df = _load_cache_safe(cfg_jdg)
    loaded_hits = len(input_keys.intersection(_pair_set(loaded_df)))

    print(
        f"\nload_cache verification: rows={len(loaded_df):,} "
        f"input hits={loaded_hits:,}/{judge_total_pairs:,}"
    )
    print("load_cache columns:", list(loaded_df.columns) if loaded_df is not None else None)

    if loaded_hits > 0:
        judge_dataframe = _filter_to_input_pairs(loaded_df, judge_input_df)
    else:
        print(
            "WARNING: _load_judge_cache did not read the assembled parquet, "
            "but judge_dataframe is already assembled and will be used directly."
        )

    judge_features_lookup = features_lookup_from_dataframe(judge_dataframe)

    # -----------------------------------------------------------------
    # Upload assembled final cache to Drive/judge/<judge_slug>
    # -----------------------------------------------------------------

    if jdc_cfg.get("enabled", True):
        final_drive_slug_dir.mkdir(parents=True, exist_ok=True)
        cmd = [
            "rsync",
            "-ah",
            "--info=progress2",
            "--partial",
            f"{local_slug_dir}/",
            f"{final_drive_slug_dir}/",
        ]
        print("\nUploading assembled final cache:")
        print(" ".join(cmd))
        proc = subprocess.run(cmd, text=True)
        if proc.returncode != 0:
            print(f"WARNING: rsync upload failed with return code {proc.returncode}")

    # -----------------------------------------------------------------
    # Final diagnostics
    # -----------------------------------------------------------------

    final_hits = len(input_keys.intersection(_pair_set(judge_dataframe)))

    p_yes_vals = (
        judge_dataframe["p_yes_renorm"].to_numpy()
        if "p_yes_renorm" in judge_dataframe.columns
        else np.zeros(0)
    )

    judge_pairs_cached_before = final_hits
    judge_pairs_newly_scored = 0
    judge_wall_time = 0.0

    print("\nFinal judge cache:")
    print(f"  rows               : {len(judge_dataframe):,}")
    print(f"  matching input pairs: {final_hits:,}/{judge_total_pairs:,}")
    print(f"  lookup entries     : {len(judge_features_lookup):,}")
    print(f"  newly scored       : 0")

    if p_yes_vals.size:
        deciles = np.quantile(p_yes_vals, np.arange(0, 11) / 10)
        print("  p_yes deciles      :", " ".join(f"{x:.3f}" for x in deciles))
        print(
            f"  p_yes mean/std     : "
            f"{float(p_yes_vals.mean()):.3f} / {float(p_yes_vals.std()):.3f}"
        )
    else:
        print("  p_yes deciles      : unavailable; p_yes_renorm column missing")

## 8d. Build the training NN index and compute NN features for train / val

We build a *full-fidelity* nearest-neighbor index over the **training**
items only (so cold-start val items aren't in the index), then precompute
the locked 8-scalar NN feature vector per training row and per val row.

Schema (8 scalars per (subject, item) prediction):

    [passrate_mean, passrate_weighted_mean, passrate_std, coverage,
     top1_label, top1_similarity, mean_similarity, n_labeled_neighbors_log1p]

The features feed *into* the residual MLP alongside pool / cluster / judge
channels. The runtime ships a separate compressed cache (PCA + int8) over
the same training items; the asymmetric design is deliberate and the
residual head absorbs the compression noise via the `coverage` feature.

Sanity check (loud): we report Pearson correlation between
`passrate_mean` and the val label. If this is near zero or negative,
*something* is mis-keyed -- likely the subject_id derivation or the
item_key -> row index map -- and you should debug before training.

In [ ]:
# ---------------------------------------------------------------------
# NN features: direct Drive cache fetch first; GPU FAISS only if needed
#
# Expected Drive cache:
#   /content/drive/MyDrive/prediction-competition-321M/<nn_cfg.cache_dir>
#
# Usually:
#   /content/drive/MyDrive/prediction-competition-321M/artifacts/nn_features
#
# Produces:
#   nn_train_matrix
#   nn_val_matrix
#   USE_NN_FEATURES
# ---------------------------------------------------------------------

import gc
import importlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from tqdm.auto import tqdm

from src.nn_features import (
    NN_FEATURE_DIM,
    NN_FEATURE_NAMES,
    NNFeaturesConfig,
    build_passrate_table,
    compute_nn_features,
)

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"[nn] Drive mount skipped/failed: {exc}", flush=True)


def _mark(msg: str) -> None:
    print(f"[nn {time.strftime('%H:%M:%S')}] {msg}", flush=True)


def _fmt_bytes(n: int) -> str:
    n = float(n)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024:
            return f"{n:.2f} {unit}"
        n /= 1024
    return f"{n:.2f} PB"


def _dir_size(path: Path) -> int:
    path = Path(path)
    if not path.exists():
        return 0
    total = 0
    for p in path.rglob("*"):
        if p.is_file():
            try:
                total += p.stat().st_size
            except OSError:
                pass
    return total


def _run(cmd, *, check=False):
    _mark("running: " + " ".join(map(str, cmd)))
    proc = subprocess.run(
        list(map(str, cmd)),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    out = proc.stdout or ""
    if out.strip():
        print(out[-5000:], flush=True)
    if check and proc.returncode != 0:
        raise RuntimeError(
            f"command failed with return code {proc.returncode}:\n"
            + " ".join(map(str, cmd))
            + "\n\n"
            + out[-5000:]
        )
    return proc


def _rsync_dir(src: Path, dst: Path) -> None:
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)

    if not src.exists():
        print(f"[nn] rsync skipped; source missing: {src}", flush=True)
        return

    cmd = [
        "rsync",
        "-ah",
        "--info=progress2",
        "--partial",
        f"{src}/",
        f"{dst}/",
    ]
    print("[nn] running:", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"rsync failed with return code {proc.returncode}: {src} -> {dst}")


def _purge_faiss_modules() -> None:
    doomed = [name for name in sys.modules if name == "faiss" or name.startswith("faiss.")]
    for name in doomed:
        try:
            del sys.modules[name]
        except Exception:
            pass
    importlib.invalidate_caches()
    gc.collect()


def _try_import_faiss():
    _purge_faiss_modules()
    try:
        import faiss
        n = faiss.get_num_gpus() if hasattr(faiss, "get_num_gpus") else 0
        print("FAISS version       :", getattr(faiss, "__version__", "unknown"), flush=True)
        print("FAISS file          :", getattr(faiss, "__file__", "unknown"), flush=True)
        print("FAISS visible GPUs  :", n, flush=True)
        return faiss, int(n)
    except Exception as exc:
        print("[nn] FAISS import failed:", repr(exc), flush=True)
        return None, 0


def _ensure_fast_gpu_faiss():
    """
    Try to get GPU FAISS in the current runtime.

    Attempt order:
      1. already-working faiss
      2. faiss-gpu-cu12-cuvs
      3. faiss-gpu-cu12[fix-cuda]
      4. faiss-gpu-cu12

    If CPU FAISS has already been imported, this still tries to purge/reimport,
    but native extension unloading is not always possible in-process.
    """
    print("Torch CUDA available:", torch.cuda.is_available(), flush=True)
    if torch.cuda.is_available():
        print("Torch CUDA version   :", torch.version.cuda, flush=True)
        print("Torch GPU            :", torch.cuda.get_device_name(0), flush=True)

    faiss, n = _try_import_faiss()
    if faiss is not None and n > 0:
        _mark("GPU FAISS already active")
        return faiss

    _mark("GPU FAISS not active; installing fast CUDA package")

    _run([
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "faiss-cpu",
        "faiss-gpu",
        "faiss-gpu-cu11",
        "faiss-gpu-cu12",
        "faiss-gpu-cu12-cuvs",
    ])

    _purge_faiss_modules()

    install_attempts = [
        ["faiss-gpu-cu12-cuvs"],
        ["faiss-gpu-cu12[fix-cuda]"],
        ["faiss-gpu-cu12"],
    ]

    last_out = ""
    for pkgs in install_attempts:
        _mark(f"installing {' '.join(pkgs)}")
        proc = _run([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--force-reinstall",
            "--no-cache-dir",
            *pkgs,
        ])
        last_out = proc.stdout or ""

        if proc.returncode != 0:
            print(f"[nn] install failed for {pkgs}; trying next option", flush=True)
            continue

        faiss, n = _try_import_faiss()
        if faiss is not None and n > 0:
            _mark(f"SUCCESS: GPU FAISS active from {' '.join(pkgs)}")
            return faiss

    raise RuntimeError(
        "Could not activate GPU FAISS in this runtime.\n"
        "The NN cache fetch may still have worked; but recomputing NN features "
        "requires GPU FAISS. If CPU FAISS was imported earlier, the only reliable "
        "fix is a runtime restart before importing faiss.\n\n"
        f"Last install output:\n{last_out[-3000:]}"
    )


def _load_nn_cache_if_valid(nn_train_npy: Path, nn_val_npy: Path, *, train_n: int, val_n: int):
    if not nn_train_npy.exists() or not nn_val_npy.exists():
        return None, None, False

    try:
        train = np.load(nn_train_npy, mmap_mode=None)
        val = np.load(nn_val_npy, mmap_mode=None)

        ok = (
            train.shape == (train_n, NN_FEATURE_DIM)
            and val.shape == (val_n, NN_FEATURE_DIM)
            and np.issubdtype(train.dtype, np.number)
            and np.issubdtype(val.dtype, np.number)
        )

        if not ok:
            print(
                "[nn] cache shape mismatch:",
                f"train={train.shape}, expected={(train_n, NN_FEATURE_DIM)};",
                f"val={val.shape}, expected={(val_n, NN_FEATURE_DIM)}",
                flush=True,
            )
            return None, None, False

        return train.astype(np.float32, copy=False), val.astype(np.float32, copy=False), True

    except Exception as exc:
        print(f"[nn] cache load failed: {repr(exc)}", flush=True)
        return None, None, False


def _save_manifest(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True))


# ---------------------------------------------------------------------
# Config + direct cache fetch
# ---------------------------------------------------------------------

NN_FEATURES_CFG_DICT = CFG.get("nn_features") or {}
nn_cfg = NNFeaturesConfig.from_dict(NN_FEATURES_CFG_DICT)
USE_NN_FEATURES = bool(nn_cfg.enabled)

NN_CACHE_DIR = ROOT / nn_cfg.cache_dir
NN_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Direct Drive path from the cache-save convention we used earlier.
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/prediction-competition-321M")
DRIVE_NN_CACHE_DIR = DRIVE_PROJECT_ROOT / nn_cfg.cache_dir

# Optional override.
if (CFG.get("nn_drive_cache") or {}).get("folder"):
    DRIVE_NN_CACHE_DIR = Path((CFG.get("nn_drive_cache") or {})["folder"])

nn_train_matrix: np.ndarray | None = None
nn_val_matrix: np.ndarray | None = None
nn_index = None
nn_passrate_csr = None
nn_passrate_mask_csr = None
nn_subject_to_id_map: dict[str, int] = {}

if not USE_NN_FEATURES:
    print("NN features disabled (CFG.nn_features.enabled = false). Skipping.", flush=True)

else:
    train_df_for_nn = splits["item_cold_start"].train
    val_df_for_nn = splits["item_cold_start"].val

    nn_train_npy = NN_CACHE_DIR / "train.npy"
    nn_val_npy = NN_CACHE_DIR / "val.npy"
    nn_subject_to_id_json = NN_CACHE_DIR / "subject_to_id.json"

    print(f"NN feature config    : k={nn_cfg.k} similarity={nn_cfg.similarity}", flush=True)
    print(f"NN local cache dir   : {NN_CACHE_DIR}", flush=True)
    print(f"NN Drive cache dir   : {DRIVE_NN_CACHE_DIR}", flush=True)
    print(f"Train rows           : {len(train_df_for_nn):,}", flush=True)
    print(f"Val rows             : {len(val_df_for_nn):,}", flush=True)

    # 1) Direct fetch cache from Drive before doing anything expensive.
    if DRIVE_NN_CACHE_DIR.exists():
        print(
            f"[nn] fetching Drive NN cache: {DRIVE_NN_CACHE_DIR} "
            f"size={_fmt_bytes(_dir_size(DRIVE_NN_CACHE_DIR))}",
            flush=True,
        )
        _rsync_dir(DRIVE_NN_CACHE_DIR, NN_CACHE_DIR)
    else:
        print(f"[nn] Drive NN cache path missing: {DRIVE_NN_CACHE_DIR}", flush=True)

    # 2) Validate local cache.
    nn_train_matrix, nn_val_matrix, cache_hit = _load_nn_cache_if_valid(
        nn_train_npy,
        nn_val_npy,
        train_n=len(train_df_for_nn),
        val_n=len(val_df_for_nn),
    )

    if cache_hit:
        print(
            "NN cache hit         : loaded train.npy + val.npy; "
            "skipping passrate, FAISS, and dataframe-heavy recompute",
            flush=True,
        )

        if nn_subject_to_id_json.exists():
            try:
                nn_subject_to_id_map = {
                    str(k): int(v)
                    for k, v in json.loads(nn_subject_to_id_json.read_text()).items()
                }
                print(f"NN subject map       : loaded {len(nn_subject_to_id_map):,} entries", flush=True)
            except Exception as exc:
                print(f"[nn] subject_to_id.json unreadable but not needed for training: {exc}", flush=True)

    else:
        # -----------------------------------------------------------------
        # Recompute only if direct cache fetch failed.
        # -----------------------------------------------------------------
        _mark("NN cache missing/incomplete; recomputing with GPU FAISS")
        faiss = _ensure_fast_gpu_faiss()

        NN_CHUNK_ROWS = int(os.environ.get("NN_CHUNK_ROWS", "65536"))
        NN_FAISS_GPU_ID = int(os.environ.get("NN_FAISS_GPU_ID", "0"))
        NN_FAISS_SEARCH_BATCH = int(os.environ.get("NN_FAISS_SEARCH_BATCH", "65536"))
        NN_FAISS_USE_FLOAT16 = bool(int(os.environ.get("NN_FAISS_USE_FLOAT16", "0")))

        print(f"NN chunk rows        : {NN_CHUNK_ROWS:,}", flush=True)
        print(f"NN FAISS GPU id      : {NN_FAISS_GPU_ID}", flush=True)
        print(f"NN FAISS search bsz  : {NN_FAISS_SEARCH_BATCH:,}", flush=True)
        print(f"NN FAISS float16     : {NN_FAISS_USE_FLOAT16}", flush=True)

        class FastGpuFaissTrainingNNIndex:
            def __init__(
                self,
                *,
                item_keys: list[str],
                embeddings: np.ndarray,
                cfg,
                cache_dir: Path,
                gpu_id: int = 0,
                search_batch_size: int = 65536,
                use_float16: bool = False,
            ):
                self.item_keys = [str(k) for k in item_keys]
                self.item_to_pos = {k: i for i, k in enumerate(self.item_keys)}
                self.cfg = cfg
                self.cache_dir = Path(cache_dir)
                self.cache_dir.mkdir(parents=True, exist_ok=True)
                self.gpu_id = int(gpu_id)
                self.search_batch_size = int(search_batch_size)
                self.use_float16 = bool(use_float16)

                x = np.ascontiguousarray(embeddings.astype(np.float32, copy=False))
                if x.ndim != 2:
                    raise ValueError(f"embeddings must be 2D, got {x.shape}")

                self.n, self.d = x.shape
                self.similarity = str(getattr(cfg, "similarity", "cosine")).lower()

                if self.similarity == "cosine":
                    faiss.normalize_L2(x)
                    metric = faiss.METRIC_INNER_PRODUCT
                    cpu_index = faiss.IndexFlatIP(self.d)
                elif self.similarity in {"l2", "euclidean"}:
                    metric = faiss.METRIC_L2
                    cpu_index = faiss.IndexFlatL2(self.d)
                else:
                    raise ValueError(f"Unsupported NN similarity: {self.similarity}")

                self._embeddings = x

                res = faiss.StandardGpuResources()
                co = faiss.GpuClonerOptions()
                co.useFloat16 = self.use_float16

                self._res = res
                self.index = faiss.index_cpu_to_gpu(res, self.gpu_id, cpu_index, co)

                _mark(f"adding {self.n:,} vectors to GPU FAISS index d={self.d}")
                self.index.add(x)
                _mark("GPU FAISS index ready")

            def _normalize_queries(self, q: np.ndarray) -> np.ndarray:
                q = np.ascontiguousarray(q.astype(np.float32, copy=False))
                if self.similarity == "cosine":
                    faiss.normalize_L2(q)
                return q

            def _search_vectors(self, q: np.ndarray, k: int):
                q = self._normalize_queries(q)
                all_idx = []
                all_sim = []

                for start in tqdm(
                    range(0, len(q), self.search_batch_size),
                    desc=f"GPU FAISS search k={k}",
                    unit="batch",
                    leave=False,
                    dynamic_ncols=True,
                ):
                    end = min(start + self.search_batch_size, len(q))
                    sims, idx = self.index.search(q[start:end], int(k))
                    all_idx.append(idx.astype(np.int64, copy=False))
                    all_sim.append(sims.astype(np.float32, copy=False))

                return np.vstack(all_idx), np.vstack(all_sim)

            def _precompute_train_neighbors(self, k: int):
                cache_path = self.cache_dir / f"train_neighbors_k{k}_exclude_self.npz"

                if cache_path.exists():
                    try:
                        z = np.load(cache_path)
                        idx = z["idx"]
                        sims = z["sims"]
                        if idx.shape == (self.n, k) and sims.shape == (self.n, k):
                            print(f"[nn] loaded cached train neighbors: {cache_path}", flush=True)
                            return idx.astype(np.int64, copy=False), sims.astype(np.float32, copy=False)
                    except Exception as exc:
                        print(f"[nn] cached train neighbors unreadable: {exc}", flush=True)

                _mark(f"precomputing train-item GPU neighbors n={self.n:,}, k={k}, exclude_self=True")

                raw_idx, raw_sims = self._search_vectors(self._embeddings, k + 1)

                idx = np.empty((self.n, k), dtype=np.int64)
                sims = np.empty((self.n, k), dtype=np.float32)

                for i in tqdm(
                    range(self.n),
                    desc="Remove self from train neighbors",
                    unit="item",
                    dynamic_ncols=True,
                ):
                    kept_i = []
                    kept_s = []
                    for j, s in zip(raw_idx[i], raw_sims[i]):
                        if int(j) == i:
                            continue
                        kept_i.append(int(j))
                        kept_s.append(float(s))
                        if len(kept_i) == k:
                            break

                    if len(kept_i) < k:
                        # Extremely unlikely; pad with best non-self available.
                        for j, s in zip(raw_idx[i], raw_sims[i]):
                            if int(j) != i:
                                while len(kept_i) < k:
                                    kept_i.append(int(j))
                                    kept_s.append(float(s))
                                break

                    idx[i] = np.asarray(kept_i[:k], dtype=np.int64)
                    sims[i] = np.asarray(kept_s[:k], dtype=np.float32)

                np.savez_compressed(cache_path, idx=idx, sims=sims)
                print(f"[nn] saved train neighbor cache: {cache_path}", flush=True)

                return idx, sims

            def nearest(self, query_embeds, k: int, exclude_self: bool = True, query_keys=None):
                q = np.asarray(query_embeds, dtype=np.float32)
                k = int(k)

                # Fast path for train rows: query_keys are known training item keys.
                if exclude_self and query_keys is not None:
                    qkeys = [str(x) for x in query_keys]
                    positions = [self.item_to_pos.get(x, -1) for x in qkeys]

                    if all(p >= 0 for p in positions):
                        base_idx, base_sims = self._precompute_train_neighbors(k)
                        pos_arr = np.asarray(positions, dtype=np.int64)
                        return base_idx[pos_arr], base_sims[pos_arr]

                # Dedupe repeated query keys to avoid repeated FAISS search.
                if query_keys is not None:
                    qkeys = [str(x) for x in query_keys]
                    first_pos = {}
                    unique_keys = []
                    unique_rows = []

                    for r, key in enumerate(qkeys):
                        if key not in first_pos:
                            first_pos[key] = len(unique_keys)
                            unique_keys.append(key)
                            unique_rows.append(r)

                    if len(unique_keys) < len(qkeys):
                        uq = q[np.asarray(unique_rows, dtype=np.int64)]
                        search_k = k + 1 if exclude_self else k
                        uidx, usims = self._search_vectors(uq, search_k)

                        out_idx = np.empty((len(qkeys), k), dtype=np.int64)
                        out_sims = np.empty((len(qkeys), k), dtype=np.float32)

                        for r, key in enumerate(qkeys):
                            ur = first_pos[key]
                            inds = uidx[ur]
                            ss = usims[ur]

                            if exclude_self and key in self.item_to_pos:
                                self_pos = self.item_to_pos[key]
                                keep = inds != self_pos
                                inds = inds[keep]
                                ss = ss[keep]

                            out_idx[r] = inds[:k]
                            out_sims[r] = ss[:k]

                        return out_idx, out_sims

                search_k = k + 1 if exclude_self else k
                idx, sims = self._search_vectors(q, search_k)

                if exclude_self and query_keys is not None:
                    out_idx = np.empty((len(q), k), dtype=np.int64)
                    out_sims = np.empty((len(q), k), dtype=np.float32)

                    for r, key in enumerate([str(x) for x in query_keys]):
                        inds = idx[r]
                        ss = sims[r]
                        if key in self.item_to_pos:
                            keep = inds != self.item_to_pos[key]
                            inds = inds[keep]
                            ss = ss[keep]
                        out_idx[r] = inds[:k]
                        out_sims[r] = ss[:k]

                    return out_idx, out_sims

                return idx[:, :k], sims[:, :k]

        # -------------------------------------------------------------
        # Build training-item index
        # -------------------------------------------------------------

        train_item_keys_full = train_df_for_nn["item_key"].astype(str).unique().tolist()
        train_item_keys_full = [k for k in train_item_keys_full if k in item_emb_lookup]

        print(f"NN index over        : {len(train_item_keys_full):,} unique training items", flush=True)

        emb_dim = int(getattr(embedder, "embedding_dim", 0))
        if emb_dim <= 0:
            first_vec = np.asarray(next(iter(item_emb_lookup.values())))
            emb_dim = int(first_vec.shape[-1])

        index_matrix_path = NN_CACHE_DIR / "gpu_faiss_index_embeddings.npy"
        index_keys_path = NN_CACHE_DIR / "gpu_faiss_index_keys.json"

        if index_matrix_path.exists() and index_keys_path.exists():
            try:
                cached_keys = json.loads(index_keys_path.read_text())
                if [str(x) for x in cached_keys] == train_item_keys_full:
                    item_emb_matrix = np.load(index_matrix_path, mmap_mode=None).astype(np.float32, copy=False)
                    if item_emb_matrix.shape != (len(train_item_keys_full), emb_dim):
                        raise ValueError(f"bad cached embedding shape {item_emb_matrix.shape}")
                    print(f"[nn] loaded cached GPU index matrix: {index_matrix_path}", flush=True)
                else:
                    raise ValueError("cached index keys differ")
            except Exception as exc:
                print(f"[nn] cached GPU index matrix unusable: {exc}; rebuilding", flush=True)
                item_emb_matrix = None
        else:
            item_emb_matrix = None

        if item_emb_matrix is None:
            item_emb_matrix = np.empty((len(train_item_keys_full), emb_dim), dtype=np.float32)

            for i, k in tqdm(
                enumerate(train_item_keys_full),
                total=len(train_item_keys_full),
                desc="Build NN index embedding matrix",
                unit="item",
                dynamic_ncols=True,
            ):
                v = np.asarray(item_emb_lookup[k], dtype=np.float32)
                if v.shape != (emb_dim,):
                    raise ValueError(f"item embedding shape for {k!r}: {v.shape}, expected {(emb_dim,)}")
                item_emb_matrix[i] = v

            np.save(index_matrix_path, item_emb_matrix)
            index_keys_path.write_text(json.dumps(train_item_keys_full))
            print(f"[nn] saved GPU index matrix: {index_matrix_path}", flush=True)

        nn_index = FastGpuFaissTrainingNNIndex(
            item_keys=train_item_keys_full,
            embeddings=item_emb_matrix,
            cfg=nn_cfg,
            cache_dir=NN_CACHE_DIR / "gpu_faiss",
            gpu_id=NN_FAISS_GPU_ID,
            search_batch_size=NN_FAISS_SEARCH_BATCH,
            use_float16=NN_FAISS_USE_FLOAT16,
        )

        # -------------------------------------------------------------
        # Pass-rate table
        # -------------------------------------------------------------

        nn_subject_to_id_map = {"<unk>": 0}
        for k0 in tqdm(
            train_df_for_nn["subject_key"].astype(str),
            total=len(train_df_for_nn),
            desc="Build subject id map",
            unit="row",
            dynamic_ncols=True,
        ):
            if k0 not in nn_subject_to_id_map:
                nn_subject_to_id_map[k0] = len(nn_subject_to_id_map)

        nn_item_index_map = {k: i for i, k in enumerate(train_item_keys_full)}

        nn_passrate_csr, nn_passrate_mask_csr = build_passrate_table(
            train_df=train_df_for_nn,
            item_index_map=nn_item_index_map,
            subject_index_map=nn_subject_to_id_map,
        )

        print(
            f"Pass-rate matrix     : shape={nn_passrate_csr.shape} "
            f"nnz={nn_passrate_csr.nnz:,} "
            f"density={nn_passrate_csr.nnz / max(1, nn_passrate_csr.shape[0] * nn_passrate_csr.shape[1]):.6f}",
            flush=True,
        )

        with open(nn_subject_to_id_json, "w", encoding="utf-8") as fh:
            json.dump(nn_subject_to_id_map, fh)

        # -------------------------------------------------------------
        # Chunked feature computation
        # -------------------------------------------------------------

        chunk_root = NN_CACHE_DIR / "resumable_gpu_faiss"
        chunk_root.mkdir(parents=True, exist_ok=True)

        def _build_query_chunk(df: pd.DataFrame, start: int, end: int):
            sub = df.iloc[start:end]
            keys = sub["item_key"].astype(str).tolist()
            subject_keys = sub["subject_key"].astype(str).tolist()

            sid = np.array(
                [nn_subject_to_id_map.get(k0, 0) for k0 in subject_keys],
                dtype=np.int64,
            )

            emb = np.empty((len(keys), emb_dim), dtype=np.float32)
            missing = 0

            for r, key in enumerate(keys):
                v = item_emb_lookup.get(key)
                if v is None:
                    emb[r] = 0.0
                    missing += 1
                else:
                    emb[r] = np.asarray(v, dtype=np.float32)

            if missing:
                print(f"[nn] WARN chunk rows {start}:{end} had {missing:,} missing item embeddings; zeroed", flush=True)

            return emb, keys, sid

        def _compute_split_chunks(df: pd.DataFrame, *, split_name: str, exclude_self: bool, out_npy: Path):
            split_chunk_dir = chunk_root / split_name
            split_chunk_dir.mkdir(parents=True, exist_ok=True)

            n = len(df)
            n_chunks = (n + NN_CHUNK_ROWS - 1) // NN_CHUNK_ROWS

            print(
                f"{split_name}: rows={n:,} chunks={n_chunks:,} "
                f"chunk_rows={NN_CHUNK_ROWS:,} exclude_self={exclude_self}",
                flush=True,
            )

            chunk_paths = []

            for ci in tqdm(
                range(n_chunks),
                desc=f"{split_name}: NN feature chunks",
                unit="chunk",
                dynamic_ncols=True,
            ):
                start = ci * NN_CHUNK_ROWS
                end = min(start + NN_CHUNK_ROWS, n)
                chunk_path = split_chunk_dir / f"chunk_{ci:05d}_{start}_{end}.npy"
                chunk_paths.append(chunk_path)

                if chunk_path.exists():
                    try:
                        arr = np.load(chunk_path, mmap_mode="r")
                        if arr.shape == (end - start, NN_FEATURE_DIM):
                            continue
                    except Exception:
                        pass

                emb, keys, sid = _build_query_chunk(df, start, end)

                feats = compute_nn_features(
                    query_embeds=emb,
                    query_item_keys=keys,
                    subject_ids=sid,
                    nn_index=nn_index,
                    passrate_csr=nn_passrate_csr,
                    passrate_mask_csr=nn_passrate_mask_csr,
                    cfg=nn_cfg,
                    exclude_self=exclude_self,
                ).astype(np.float32, copy=False)

                if feats.shape != (end - start, NN_FEATURE_DIM):
                    raise ValueError(
                        f"{split_name} chunk {ci} produced shape {feats.shape}, "
                        f"expected {(end - start, NN_FEATURE_DIM)}"
                    )

                np.save(chunk_path, feats)

                del emb, feats
                gc.collect()

            # Assemble final matrix.
            out = np.empty((n, NN_FEATURE_DIM), dtype=np.float32)

            for ci, chunk_path in enumerate(tqdm(
                chunk_paths,
                desc=f"{split_name}: assemble matrix",
                unit="chunk",
                dynamic_ncols=True,
            )):
                start = ci * NN_CHUNK_ROWS
                arr = np.load(chunk_path)
                out[start:start + len(arr)] = arr

            np.save(out_npy, out)
            print(
                f"{split_name}: saved {out_npy} shape={out.shape} "
                f"size={_fmt_bytes(out_npy.stat().st_size)}",
                flush=True,
            )
            return out

        nn_train_matrix = _compute_split_chunks(
            train_df_for_nn,
            split_name="train",
            exclude_self=True,
            out_npy=nn_train_npy,
        )

        nn_val_matrix = _compute_split_chunks(
            val_df_for_nn,
            split_name="val",
            exclude_self=False,
            out_npy=nn_val_npy,
        )

        # Upload newly computed cache back to Drive.
        DRIVE_NN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        _save_manifest(
            NN_CACHE_DIR / "gpu_faiss_manifest.json",
            {
                "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                "nn_k": int(nn_cfg.k),
                "similarity": str(nn_cfg.similarity),
                "train_shape": list(nn_train_matrix.shape),
                "val_shape": list(nn_val_matrix.shape),
                "nn_feature_dim": int(NN_FEATURE_DIM),
                "faiss_gpu_package": "faiss-gpu-cu12-cuvs preferred; fallback faiss-gpu-cu12",
            },
        )
        _rsync_dir(NN_CACHE_DIR, DRIVE_NN_CACHE_DIR)

    # -----------------------------------------------------------------
    # Sanity checks
    # -----------------------------------------------------------------

    assert nn_train_matrix is not None
    assert nn_val_matrix is not None

    if nn_train_matrix.shape != (len(train_df_for_nn), NN_FEATURE_DIM):
        raise ValueError(
            f"bad nn_train_matrix shape={nn_train_matrix.shape}, "
            f"expected={(len(train_df_for_nn), NN_FEATURE_DIM)}"
        )

    if nn_val_matrix.shape != (len(val_df_for_nn), NN_FEATURE_DIM):
        raise ValueError(
            f"bad nn_val_matrix shape={nn_val_matrix.shape}, "
            f"expected={(len(val_df_for_nn), NN_FEATURE_DIM)}"
        )

    pmean = nn_val_matrix[:, 0].astype(np.float64)
    coverage = nn_val_matrix[:, 3].astype(np.float64)
    y_val_for_corr = val_df_for_nn["label"].astype(float).to_numpy()

    corr = float("nan")
    if pmean.std() > 1e-9 and y_val_for_corr.std() > 1e-9:
        corr = float(np.corrcoef(pmean, y_val_for_corr)[0, 1])

    cov_pos = float((coverage > 0).mean())

    print(
        f"NN sanity (val)      : corr(passrate_mean, label)={corr:.4f} | "
        f"coverage>0 fraction={cov_pos:.3f}",
        flush=True,
    )
    print(
        f"NN top-K mean sim    : train mean={float(nn_train_matrix[:, 6].mean()):.4f} "
        f"std={float(nn_train_matrix[:, 6].std()):.4f}",
        flush=True,
    )

    if not np.isfinite(corr) or corr <= 0.0:
        print(
            "  WARN: passrate_mean does not correlate positively with val label. "
            "This can be real, but usually means subject/item keys should be checked.",
            flush=True,
        )

    print("NN features ready    :")
    print("  nn_train_matrix:", nn_train_matrix.shape, nn_train_matrix.dtype)
    print("  nn_val_matrix  :", nn_val_matrix.shape, nn_val_matrix.dtype)
    print("  local cache     :", NN_CACHE_DIR)
    print("  drive cache     :", DRIVE_NN_CACHE_DIR)

## 9. Embedding sanity checks

In [ ]:
from src.sanity_checks import (
    check_embedding_determinism,
    check_embedding_nan_inf,
    check_embedding_shape,
    check_embedding_truncation,
)

all_item_emb = np.stack(list(item_emb_lookup.values()), axis=0)
all_subject_emb = np.stack(list(subject_emb_lookup.values()), axis=0)
embed_checks = [
    check_embedding_shape(embedder),
    check_embedding_nan_inf(all_item_emb),
    check_embedding_nan_inf(all_subject_emb),
    check_embedding_truncation(emb_stats),
    check_embedding_determinism(embedder),
]
print_results(embed_checks)

## 10. Build the indexer and the training matrices

Subject + benchmark-condition keys -> integer ids. Index 0 is UNK in both
spaces; test-time predict() will route unseen keys there.

In [ ]:
from src.models import Indexer, LookupDataset, ModelConfig
from src.embeddings import stack_lookup
from src.item_features import build_feature_matrix
from src.judge import build_judge_matrix


def _pool_matrix(keys, features_df):
    """Build a [N, pool_feature_dim] z-scored matrix for the given item keys.

    Returns ``None`` if pool features are disabled / missing.
    """
    if features_df is None:
        return None
    return build_feature_matrix(
        [str(k) for k in keys],
        features_df,
        feature_cols=list(POOL_FEATURE_NAMES),
        key_col="item_key",
    )


def _cluster_vector(keys, assignments):
    """Build a [N] int64 cluster-id vector. Unknown keys map to 0 (UNK)."""
    if assignments is None:
        return None
    return np.array(
        [int(assignments.get(str(k), 0)) for k in keys], dtype=np.int64
    )


def _judge_matrix(subject_keys, item_keys, lookup):
    """Build the [N, 4] judge-feature matrix; missing pairs get zeros.

    Unknown ``(subject_key, item_key)`` pairs (very rare in training data,
    possible in val if val has pairs the judge step didn't see) fall back
    to the zero vector. The residual MLP's input LayerNorm absorbs the
    shift, and the model can learn to recognize the "no judge data"
    fingerprint.
    """
    if not lookup:
        return None
    return build_judge_matrix(
        [str(k) for k in subject_keys],
        [str(k) for k in item_keys],
        lookup,
    )


def _build_arrays(
    split_art,
    indexer,
    item_lookup,
    subject_lookup,
    *,
    use_subject_emb: bool = False,
    pool_features_z=None,
    cluster_assignments=None,
    judge_lookup=None,
    nn_train: np.ndarray | None = None,
    nn_val: np.ndarray | None = None,
):
    train = split_art.train
    val = split_art.val
    s_train = np.array([indexer.subject_id(k) for k in train["subject_key"]])
    s_val = np.array([indexer.subject_id(k) for k in val["subject_key"]])
    bc_train = np.array([indexer.bc_id(k) for k in train["benchmark_condition_key"]])
    bc_val = np.array([indexer.bc_id(k) for k in val["benchmark_condition_key"]])
    ie_train = stack_lookup(train["item_key"], item_lookup)
    ie_val = stack_lookup(val["item_key"], item_lookup)
    se_train = None
    se_val = None
    if use_subject_emb:
        se_train = stack_lookup(train["subject_key"], subject_lookup)
        se_val = stack_lookup(val["subject_key"], subject_lookup)
    pf_train = _pool_matrix(train["item_key"], pool_features_z)
    pf_val = _pool_matrix(val["item_key"], pool_features_z)
    ci_train = _cluster_vector(train["item_key"], cluster_assignments)
    ci_val = _cluster_vector(val["item_key"], cluster_assignments)
    jf_train = _judge_matrix(train["subject_key"], train["item_key"], judge_lookup or {})
    jf_val = _judge_matrix(val["subject_key"], val["item_key"], judge_lookup or {})
    # NN features are pre-computed once in cell 8d so the per-batch step
    # never re-hits FAISS or the sparse passrate matrix.
    nn_train_arr = (
        np.asarray(nn_train, dtype=np.float32) if nn_train is not None else None
    )
    nn_val_arr = (
        np.asarray(nn_val, dtype=np.float32) if nn_val is not None else None
    )
    if nn_train_arr is not None and nn_train_arr.shape[0] != len(train):
        raise ValueError(
            f"nn_train rows={nn_train_arr.shape[0]} != train rows={len(train)}"
        )
    if nn_val_arr is not None and nn_val_arr.shape[0] != len(val):
        raise ValueError(
            f"nn_val rows={nn_val_arr.shape[0]} != val rows={len(val)}"
        )
    y_train = train["label"].astype(float).to_numpy()
    y_val = val["label"].astype(float).to_numpy()
    return (
        LookupDataset(
            s_train, bc_train, ie_train, y_train, se_train, pf_train,
            ci_train, jf_train, nn_train_arr,
        ),
        LookupDataset(
            s_val, bc_val, ie_val, y_val, se_val, pf_val,
            ci_val, jf_val, nn_val_arr,
        ),
    )


primary = splits["item_cold_start"]
indexer = Indexer.fit(
    subject_keys=primary.train["subject_key"].tolist(),
    bc_keys=primary.train["benchmark_condition_key"].tolist(),
)
print(f"Indexer: n_subjects={indexer.n_subjects}  n_bc={indexer.n_bc}")

USE_SUBJECT_EMB = False  # set True to feed subject text embeddings to the residual
SUBJECT_EMB_DIM = embedder.embedding_dim if USE_SUBJECT_EMB else 0
ITEM_EMB_DIM = embedder.embedding_dim

# Fit z-score stats on the TRAIN items only, then apply to all items.
if USE_POOL_FEATURES and pool_df is not None:
    train_item_keys = set(primary.train["item_key"].astype(str).tolist())
    train_features = pool_df[pool_df["item_key"].astype(str).isin(train_item_keys)]
    pool_stats = load_zscore_stats(POOL_STATS_PATH)
    if pool_stats is None:
        pool_stats = fit_zscore_stats(train_features, feature_cols=list(POOL_FEATURE_NAMES))
        save_zscore_stats(pool_stats, POOL_STATS_PATH)
        print(f"Fit pool-feature z-score stats on {len(train_features):,} train items -> {POOL_STATS_PATH.relative_to(ROOT)}")
    else:
        print(f"Loaded pool-feature z-score stats from {POOL_STATS_PATH.relative_to(ROOT)}")
    pool_features_z = apply_zscore(pool_df, pool_stats)
else:
    pool_stats = None
    pool_features_z = None

USE_JUDGE_FEATURES = bool(
    JUDGE_ENABLED and JUDGE_FEATURES_IN_RESIDUAL and judge_features_lookup
)
# Use NN features iff we built them in cell 8d and the config says so.
USE_NN_FEATURES = bool(
    USE_NN_FEATURES and nn_train_matrix is not None and nn_val_matrix is not None
)

train_ds, val_ds = _build_arrays(
    primary,
    indexer,
    item_emb_lookup,
    subject_emb_lookup,
    use_subject_emb=USE_SUBJECT_EMB,
    pool_features_z=pool_features_z,
    cluster_assignments=cluster_assignments,
    judge_lookup=judge_features_lookup if USE_JUDGE_FEATURES else None,
    nn_train=nn_train_matrix if USE_NN_FEATURES else None,
    nn_val=nn_val_matrix if USE_NN_FEATURES else None,
)
print(
    f"train rows: {len(train_ds)} | val rows: {len(val_ds)} | "
    f"pool_feats: {'on' if USE_POOL_FEATURES else 'off'} | "
    f"clusters: {'on' if USE_CLUSTER_FEATURES else 'off'} | "
    f"judge: {'on' if USE_JUDGE_FEATURES else 'off'} | "
    f"nn: {'on' if USE_NN_FEATURES else 'off'}"
)

## 11. Model sanity checks: forward pass, tiny-batch overfit, random labels

In [ ]:
from src.models import build_model
from src.sanity_checks import (
    check_forward_pass,
    check_overfit_tiny_batch,
    check_random_labels_sanity,
)


def _model_cfg(
    k: int,
    model_name: str,
    *,
    use_pool: bool | None = None,
    use_clusters: bool | None = None,
    use_nn: bool | None = None,
) -> ModelConfig:
    """Construct a ModelConfig honoring the pool / cluster / nn flags.

    ``use_pool`` / ``use_clusters`` / ``use_nn`` default to the global
    notebook flags; pass explicit booleans in the ablation grid to toggle
    them per run.
    """
    irt_reg = (CFG["train"].get("irt_reg") or {})
    up = USE_POOL_FEATURES if use_pool is None else bool(use_pool)
    uc = USE_CLUSTER_FEATURES if use_clusters is None else bool(use_clusters)
    uj = bool(USE_JUDGE_FEATURES)
    un = USE_NN_FEATURES if use_nn is None else bool(use_nn)
    return ModelConfig(
        k=k,
        item_embed_dim=ITEM_EMB_DIM,
        item_map_hidden_dim=int(CFG["train"]["item_map_hidden_dim"]),
        residual_hidden_dim=int(CFG["train"]["residual_hidden_dim"]),
        dropout=float(CFG["train"]["dropout"]),
        n_subjects=indexer.n_subjects,
        n_benchmark_conditions=indexer.n_bc,
        use_subject_text_embedding=USE_SUBJECT_EMB,
        subject_embed_dim=SUBJECT_EMB_DIM,
        lambda_resid_init=float(CFG["train"]["lambda_resid_init"]),
        lambda_resid_trainable=bool(CFG["train"]["lambda_resid_trainable"]),
        use_pool_features=bool(up and pool_features_z is not None),
        pool_feature_dim=POOL_FEATURE_DIM if up else 0,
        use_cluster_features=bool(uc and cluster_assignments is not None),
        n_clusters=N_CLUSTERS if uc else 0,
        cluster_embed_dim=CLUSTER_EMBED_DIM if uc else 0,
        irt_lambda_beta=float(irt_reg.get("lambda_beta", 1.0e-4)),
        irt_lambda_alpha=float(irt_reg.get("lambda_alpha", 1.0e-4)),
        use_judge_features=bool(uj),
        judge_feature_dim=int(JUDGE_FEATURE_DIM) if uj else 0,
        use_nn_features=bool(un),
        nn_feature_dim=int(NN_FEATURE_DIM) if un else 0,
    )


SMOKE_K = int(CFG["train"]["k_factors"][0])
smoke_model_cfg = _model_cfg(SMOKE_K, "kfactor")
forward_result = check_forward_pass(
    build_model("kfactor", smoke_model_cfg),
    item_emb_dim=ITEM_EMB_DIM,
    n_subjects=indexer.n_subjects,
    n_bc=indexer.n_bc,
    subject_emb_dim=SUBJECT_EMB_DIM,
)
overfit_result = check_overfit_tiny_batch(
    build_model("kfactor", smoke_model_cfg),
    item_emb_dim=ITEM_EMB_DIM,
    n_subjects=indexer.n_subjects,
    n_bc=indexer.n_bc,
    subject_emb_dim=SUBJECT_EMB_DIM,
)
random_label_result = check_random_labels_sanity(
    lambda: build_model("kfactor", smoke_model_cfg),
    item_emb_dim=ITEM_EMB_DIM,
    n_subjects=indexer.n_subjects,
    n_bc=indexer.n_bc,
)
print_results([forward_result, overfit_result, random_label_result])

## 12. Baselines (computed before training the heavy models)

Global mean / subject-shrinkage / benchmark-condition shrinkage / logistic
on raw embeddings. Saved into the per-split results table.

In [ ]:
from src.eval import (
    bc_mean_with_shrinkage,
    compute_metrics,
    global_mean_baseline,
    logistic_baseline_on_embeddings_streaming,
    subject_mean_with_shrinkage,
)

per_run_predictions: dict[str, dict[str, tuple[np.ndarray, np.ndarray]]] = {}
all_results: list[dict] = []


def _add_baseline(name: str, split_name: str, y_val: np.ndarray, p_val: np.ndarray):
    m = compute_metrics(y_val, p_val, n_bins=int(CFG["eval"]["ece_bins"]))
    row = {
        "model_name": name,
        "split": split_name,
        "k": -1,
        "seed": 0,
        "val_log_loss": m.log_loss,
        "val_brier": m.brier,
        "val_auc": m.auc,
        "val_accuracy": m.accuracy,
        "val_ece": m.ece,
        "n_val": m.n,
    }
    all_results.append(row)
    per_run_predictions.setdefault(split_name, {})[name] = (y_val, p_val)


# Cheap baselines (every split).
for split_name, art in splits.items():
    y_val = art.val["label"].to_numpy().astype(float)
    y_train = art.train["label"].to_numpy().astype(float)

    p_gm = global_mean_baseline(y_train, y_val)
    _add_baseline("baseline_global_mean", split_name, y_val, p_gm)

    p_sub = subject_mean_with_shrinkage(art.train, art.val, alpha=20.0)
    _add_baseline("baseline_subject_shrinkage", split_name, y_val, p_sub)

    p_bc = bc_mean_with_shrinkage(art.train, art.val, alpha=20.0)
    _add_baseline("baseline_bc_shrinkage", split_name, y_val, p_bc)

# Memory-safe streaming logistic baseline on item embeddings (primary split).
# Avoids materializing [n_rows, embedding_dim] -- important for high-dim
# encoders like Qwen3-Embedding-4B (d=2560) on multi-million-row datasets.
y_val = primary.val["label"].to_numpy().astype(float)
p_log = logistic_baseline_on_embeddings_streaming(
    primary.train,
    primary.val,
    item_emb_lookup,
    batch_size=16384,        # bump to 65536 if GPU is underutilized
    epochs=3,                # baseline only; do not over-invest
    lr=1e-3,
    weight_decay=1e-4,
    bf16=bool(CFG["encoder"]["bf16"]),
)
_add_baseline("baseline_logistic_items_streaming", "item_cold_start", y_val, p_log)

baseline_df = pd.DataFrame(all_results).sort_values(["split", "val_log_loss"])
print(baseline_df.to_string(index=False))

## 13. Extended ablation grid

Trains every configured model variant across the configured ``k_factors``
and ``seeds``. The three new variants (``kfactor_irt_item``,
``kfactor_irt_item_mlp``, ``kfactor_irt_item_gated_mlp``) add a parallel
Item-IRT channel: ``logit = alpha(item) * (theta_subj - beta(item))`` plus
the existing offsets and (optionally) a residual MLP that can also see
pool features and cluster embeddings.

When ``RUN_FEATURE_TOGGLE_GRID`` is true we also re-run the variants that
*can* consume pool / cluster features with those channels off, so the
diagnostic in cell 14b can attribute gains cleanly. Set it to ``False``
for fast iteration.

Saves best checkpoint per run by item-cold-start val log-loss. The trainer
streams JSONL progress events to ``PROGRESS_FILE`` so you can tail it from
another shell.

In [ ]:
import subprocess
import logging
import time
import sys
import html as _html
from dataclasses import asdict

import pandas as pd
import torch
from IPython.display import display, HTML

import src.train as train_mod
from src.train import TrainConfig, train_one


# ---------------------------------------------------------------------------
# Robust Colab-visible tqdm replacement
# ---------------------------------------------------------------------------
# This avoids relying on tqdm.notebook / tqdm.auto widget rendering, which can
# silently fail in Colab, especially for nested progress bars.
#
# It supports the methods used by this notebook and src.train:
#   tqdm(...)
#   pbar.set_description(...)
#   pbar.set_postfix(...)
#   pbar.update(0)
#   pbar.close()
#
# We patch both:
#   1. local tqdm used by this cell's outer job loop
#   2. train_mod.tqdm used inside train_one() for train/val bars

class ColabVisibleTqdm:
    def __init__(
        self,
        iterable=None,
        total=None,
        desc=None,
        unit="it",
        leave=True,
        disable=False,
        mininterval=0.25,
        dynamic_ncols=True,
        **kwargs,
    ):
        self.iterable = iterable
        self.desc = str(desc or "")
        self.unit = unit or "it"
        self.leave = True
        self.disable = bool(disable)
        self.mininterval = float(mininterval)
        self.n = 0
        self.postfix = {}
        self.start_time = time.time()
        self.last_render = 0.0

        if total is None and iterable is not None:
            try:
                total = len(iterable)
            except Exception:
                total = None
        self.total = total

        self._handle = None
        if not self.disable:
            self._handle = display(HTML(self._render_html()), display_id=True)

    def __iter__(self):
        if self.iterable is None:
            return
        for item in self.iterable:
            self.n += 1
            self.refresh()
            yield item
        self.refresh(force=True)

    def update(self, n=1):
        self.n += int(n)
        self.refresh(force=(n == 0))

    def set_description(self, desc=None, refresh=True):
        self.desc = str(desc or "")
        if refresh:
            self.refresh(force=True)

    def set_postfix(self, ordered_dict=None, refresh=True, **kwargs):
        data = {}
        if ordered_dict:
            data.update(dict(ordered_dict))
        data.update(kwargs)
        self.postfix = data
        if refresh:
            self.refresh(force=True)

    def refresh(self, force=False):
        if self.disable or self._handle is None:
            return

        now = time.time()
        if not force and (now - self.last_render) < self.mininterval:
            return

        self.last_render = now
        self._handle.update(HTML(self._render_html()))

    def close(self):
        self.refresh(force=True)

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        self.close()
        return False

    def _render_html(self):
        elapsed = max(time.time() - self.start_time, 1e-9)
        rate = self.n / elapsed

        desc = _html.escape(self.desc)

        if self.total:
            pct = min(100.0, 100.0 * self.n / max(1, self.total))
            counter = f"{self.n:,}/{self.total:,}"
            progress = (
                f"<progress value='{self.n}' max='{self.total}' "
                f"style='width:460px;height:18px;vertical-align:middle;'></progress> "
                f"{pct:5.1f}%"
            )
        else:
            counter = f"{self.n:,}"
            progress = f"<span>{counter}</span>"

        postfix = ""
        if self.postfix:
            postfix = " | " + " | ".join(
                f"{_html.escape(str(k))}={_html.escape(str(v))}"
                for k, v in self.postfix.items()
            )

        return f"""
        <div style="
            font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace;
            font-size: 13px;
            line-height: 1.55;
            margin: 6px 0;
            white-space: nowrap;
        ">
            <b>{desc}</b><br>
            {progress}
            &nbsp; {counter} {self.unit}
            &nbsp; {rate:,.1f} {self.unit}/s
            {postfix}
        </div>
        """


# Patch local and trainer tqdm.
tqdm = ColabVisibleTqdm
train_mod.tqdm = ColabVisibleTqdm


# ---------------------------------------------------------------------------
# Logging / GPU diagnostics
# ---------------------------------------------------------------------------

logging.getLogger("train").setLevel(logging.INFO)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), flush=True)
    print("CUDA:", torch.version.cuda, flush=True)
    print("bf16 supported:", torch.cuda.is_bf16_supported(), flush=True)
    torch.set_float32_matmul_precision("high")
else:
    print("WARNING: CUDA not available. Training will be slow.", flush=True)


def fmt_seconds(seconds: float) -> str:
    seconds = int(max(0, seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}h {m}m {s}s"
    if m:
        return f"{m}m {s}s"
    return f"{s}s"


def gpu_status() -> str:
    try:
        out = subprocess.check_output(
            [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw",
                "--format=csv,noheader,nounits",
            ],
            text=True,
        ).strip()
        util, mem_used, mem_total, power = [x.strip() for x in out.split(",")[:4]]
        return f"GPU util={util}% mem={mem_used}/{mem_total}MB power={power}W"
    except Exception:
        return "GPU status unavailable"


# ---------------------------------------------------------------------------
# Paths / train config
# ---------------------------------------------------------------------------

CKPT_DIR = ROOT / CFG["eval"]["checkpoints_dir"]
CKPT_DIR.mkdir(parents=True, exist_ok=True)

PROGRESS_FILE = ROOT / "artifacts" / "training_progress.jsonl"
PROGRESS_FILE.parent.mkdir(parents=True, exist_ok=True)

# Clear old JSONL progress so live monitoring is not confused by stale events.
PROGRESS_FILE.write_text("")

train_cfg = TrainConfig(
    learning_rate=float(CFG["train"]["learning_rate"]),
    weight_decay=float(CFG["train"]["weight_decay"]),
    batch_size=int(CFG["train"]["batch_size"]),
    epochs=int(CFG["train"]["epochs"]),
    warmup_steps=int(CFG["train"]["warmup_steps"]),
    scheduler=str(CFG["train"]["scheduler"]),
    grad_clip=float(CFG["train"]["grad_clip"]),
    early_stopping_patience=int(CFG["train"]["early_stopping_patience"]),
    bf16=bool(CFG["encoder"]["bf16"]),
    num_workers=int(CFG["train"].get("num_workers", 0)),
    progress=True,
    log_every_batches=1,
    progress_file=str(PROGRESS_FILE),
)

print("Train config:", flush=True)
print(asdict(train_cfg), flush=True)
print(gpu_status(), flush=True)
print("Progress file:", PROGRESS_FILE, flush=True)


# ---------------------------------------------------------------------------
# Ablation grid
# ---------------------------------------------------------------------------

models_to_train = list(CFG["train"]["models"])
ks = [int(k) for k in CFG["train"]["k_factors"]]
seeds = [int(s) for s in CFG["train"]["seeds"]]

RUN_FEATURE_TOGGLE_GRID = False

_MLP_VARIANTS = {
    "kfactor_mlp",
    "kfactor_gated_mlp",
    "kfactor_irt_item_mlp",
    "kfactor_irt_item_gated_mlp",
}


def _feature_toggles_for(model_name: str) -> list[tuple[str, bool, bool]]:
    """Return a list of (tag, use_pool, use_clusters) to train for this model."""
    if not RUN_FEATURE_TOGGLE_GRID or model_name not in _MLP_VARIANTS:
        return [("full", USE_POOL_FEATURES, USE_CLUSTER_FEATURES)]
    return [
        ("full", USE_POOL_FEATURES, USE_CLUSTER_FEATURES),
        ("nofeat", False, False),
    ]


jobs = [
    (model_name, k, seed, tag, up, uc)
    for model_name in models_to_train
    for k in ks
    for seed in seeds
    for (tag, up, uc) in _feature_toggles_for(model_name)
]

print(f"\nTotal jobs: {len(jobs)}", flush=True)
print("Models:", models_to_train, flush=True)
print("k values:", ks, flush=True)
print("Seeds:", seeds, flush=True)
print(f"Feature-toggle grid: {'on' if RUN_FEATURE_TOGGLE_GRID else 'off'}", flush=True)


# ---------------------------------------------------------------------------
# Run training
# ---------------------------------------------------------------------------

ALL_RUNS: list[dict] = []
completed_times: list[float] = []
global_t0 = time.time()

pbar = tqdm(jobs, desc="Extended ablation grid", unit="run")

for job_idx, (model_name, k, seed, tag, up, uc) in enumerate(pbar, start=1):
    run_id = f"{model_name}_k{k}_seed{seed}_{tag}"
    model_cfg = _model_cfg(k, model_name, use_pool=up, use_clusters=uc)

    pbar.set_description(f"{model_name} k={k} seed={seed} {tag}")

    print("\n" + "=" * 100, flush=True)
    print(f"Starting {job_idx}/{len(jobs)}: {run_id}", flush=True)
    print("Model config:", flush=True)
    print(asdict(model_cfg), flush=True)
    print(gpu_status(), flush=True)

    job_t0 = time.time()

    r = train_one(
        model_name=model_name,
        model_cfg=model_cfg,
        train_cfg=train_cfg,
        train_ds=train_ds,
        val_ds=val_ds,
        indexer=indexer,
        seed=seed,
        run_id=run_id,
        checkpoint_dir=CKPT_DIR,
        extra_metadata={
            "encoder_model_id": CFG["encoder"]["model_id"],
            "use_subject_text_embedding": USE_SUBJECT_EMB,
            "use_pool_features": bool(model_cfg.use_pool_features),
            "use_cluster_features": bool(model_cfg.use_cluster_features),
            "use_judge_features": bool(getattr(model_cfg, "use_judge_features", False)),
            "use_nn_features": bool(getattr(model_cfg, "use_nn_features", False)),
            "feature_tag": tag,
        },
    )

    job_elapsed = time.time() - job_t0
    completed_times.append(job_elapsed)

    ALL_RUNS.append(
        {
            "run_id": r.run_id,
            "model_name": r.model_name,
            "k": r.k,
            "seed": r.seed,
            "feature_tag": tag,
            "use_pool_features": bool(model_cfg.use_pool_features),
            "use_cluster_features": bool(model_cfg.use_cluster_features),
            "use_judge_features": bool(getattr(model_cfg, "use_judge_features", False)),
            "use_nn_features": bool(getattr(model_cfg, "use_nn_features", False)),
            "epoch_best": r.epoch_best,
            "best_val_log_loss": r.best_val_log_loss,
            "best_val_brier": r.best_val_brier,
            "best_val_auc": r.best_val_auc,
            "checkpoint_path": r.checkpoint_path,
            "metadata_path": r.metadata_path,
            "elapsed_seconds": r.elapsed_seconds,
        }
    )

    avg_time = sum(completed_times) / len(completed_times)
    remaining_runs = len(jobs) - job_idx
    eta = avg_time * remaining_runs
    total_elapsed = time.time() - global_t0

    pbar.set_postfix(
        {
            "last": fmt_seconds(job_elapsed),
            "avg": fmt_seconds(avg_time),
            "ETA": fmt_seconds(eta),
            "best_ll": f"{r.best_val_log_loss:.5f}",
        }
    )
    pbar.update(0)

    runs_df_live = pd.DataFrame(ALL_RUNS).sort_values(
        "best_val_log_loss", ascending=True
    )

    print(f"\nFinished {job_idx}/{len(jobs)}: {run_id}", flush=True)
    print(f"Run time: {fmt_seconds(job_elapsed)}", flush=True)
    print(f"Total elapsed: {fmt_seconds(total_elapsed)}", flush=True)
    print(f"Estimated remaining: {fmt_seconds(eta)}", flush=True)
    print(f"Best epoch: {r.epoch_best}", flush=True)
    print(
        f"Best val log-loss: {r.best_val_log_loss:.6f} | "
        f"Brier: {r.best_val_brier:.6f} | "
        f"AUC: {r.best_val_auc if r.best_val_auc is not None else 'n/a'}",
        flush=True,
    )
    print(gpu_status(), flush=True)
    print("\nCurrent top runs:", flush=True)
    print(runs_df_live.head(10).to_string(index=False), flush=True)

pbar.close()

runs_df = pd.DataFrame(ALL_RUNS).sort_values("best_val_log_loss", ascending=True)

print("\n=== Extended ablation grid sorted by item-cold-start val log-loss ===", flush=True)
print(runs_df.to_string(index=False), flush=True)
print(f"\nTotal grid time: {fmt_seconds(time.time() - global_t0)}", flush=True)

In [ ]:
!tail -n 20 /content/prediction-competition-321M/artifacts/training_progress.jsonl

## 14. Evaluate trained checkpoints on every split + slicewise metrics

Builds the canonical results table and writes ``artifacts/results/results.csv``.

In [ ]:
from src.eval import (
    attach_subject_family,
    build_results_dataframe,
    metrics_by_group,
    metrics_by_token_length,
)
from src.train import evaluate_model

RESULTS_PATH = ROOT / CFG["eval"]["results_path"]
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)


def _load_for_inference(ckpt_path: Path):
    ck = torch.load(ckpt_path, map_location="cpu")
    cfg_dict = dict(ck["model_cfg"])
    model_cfg_obj = ModelConfig(**cfg_dict)
    return ck, model_cfg_obj


per_run_split_predictions: dict[str, dict[str, tuple[np.ndarray, np.ndarray]]] = {}


def _predict_with_checkpoint(model_name: str, ckpt_path: Path, split_art) -> np.ndarray:
    ck, mcfg = _load_for_inference(ckpt_path)
    mdl = build_model(model_name, mcfg)
    mdl.load_state_dict(ck["model_state"], strict=False)
    # Mirror the checkpoint's feature-channel config when building the val
    # dataset so pool / cluster / judge / nn channels line up with the
    # trained model.
    pf_z = pool_features_z if getattr(mcfg, "use_pool_features", False) else None
    ca = cluster_assignments if getattr(mcfg, "use_cluster_features", False) else None
    jl = (
        judge_features_lookup
        if getattr(mcfg, "use_judge_features", False)
        else None
    )
    # For NN features we only have a precomputed matrix for the primary
    # item-cold-start split. Other splits compute zero NN features (the
    # model's input LayerNorm absorbs the shift) -- the diagnostic in
    # cell 14b uses the primary split, which has the real NN matrix.
    nn_t = (
        nn_train_matrix if getattr(mcfg, "use_nn_features", False) and split_art is splits["item_cold_start"]
        else None
    )
    nn_v = (
        nn_val_matrix if getattr(mcfg, "use_nn_features", False) and split_art is splits["item_cold_start"]
        else None
    )
    val_ds_split = _build_arrays(
        split_art,
        indexer,
        item_emb_lookup,
        subject_emb_lookup,
        use_subject_emb=USE_SUBJECT_EMB,
        pool_features_z=pf_z,
        cluster_assignments=ca,
        judge_lookup=jl,
        nn_train=nn_t,
        nn_val=nn_v,
    )[1]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    eval_bs = max(int(CFG["train"]["batch_size"]), 4096)
    ll, brier_val, auc_val, p_val, y_val = evaluate_model(
        mdl, val_ds_split, device=device, batch_size=eval_bs, bf16=bool(CFG["encoder"]["bf16"])
    )
    return p_val, y_val


for r in ALL_RUNS:
    ckpt = Path(r["checkpoint_path"])
    if not ckpt.exists():
        continue
    for split_name, art in splits.items():
        # Build per-split dataset using THIS split's train/val
        if split_name != "item_cold_start":
            split_indexer = Indexer.fit(
                subject_keys=art.train["subject_key"].tolist(),
                bc_keys=art.train["benchmark_condition_key"].tolist(),
            )
            # NOTE: we only EVAL the existing checkpoint, which was trained on
            # the item_cold_start split. Indices may not align if subjects /
            # bc don't overlap; we therefore evaluate using the ORIGINAL
            # indexer (mapping new keys to UNK).
            pass
        p_val, y_val = _predict_with_checkpoint(r["model_name"], ckpt, art)
        m = compute_metrics(y_val, p_val, n_bins=int(CFG["eval"]["ece_bins"]))
        row = {
            "model_name": r["model_name"],
            "run_id": r["run_id"],
            "split": split_name,
            "k": r["k"],
            "seed": r["seed"],
            "val_log_loss": m.log_loss,
            "val_brier": m.brier,
            "val_auc": m.auc,
            "val_accuracy": m.accuracy,
            "val_ece": m.ece,
            "n_val": m.n,
        }
        all_results.append(row)
        per_run_split_predictions.setdefault(split_name, {})[r["run_id"]] = (y_val, p_val)

results_df = build_results_dataframe(
    all_results, primary_split="item_cold_start", primary_metric="val_log_loss"
)
results_df.to_csv(RESULTS_PATH, index=False)
print(f"Wrote {RESULTS_PATH.relative_to(ROOT)}")
print(results_df.to_string(index=False))

## 14b. Feature contribution and component decomposition (diagnostic)

Two analyses on the *item-cold-start val split* for the best-performing
trained model:

**Analysis A -- Leave-one-out feature ablation.** For each channel in the
model (pool features, cluster embedding, IRT alpha, IRT beta, residual
MLP, and each individual pool feature), we zero / clamp that channel at
inference and record val NLL / AUC vs. the unablated baseline. The
inference-only ablation is fast and directionally honest: it shows how
much *the trained model relies on* each channel today. To upgrade to a
"retrain only the residual head" estimate (more accurate when channels
strongly interact), retrain the residual MLP with the same masks applied
during training -- the helpers below take a model factory so wrapping
that loop is straightforward.

**Analysis B -- Logit-component decomposition.** For each val example we
decompose the final logit into its additive components (IRT, offsets,
MLP) and report Var, Pearson(c_i, y), Solo NLL (a 2-param logistic on c_i
alone, so the metric reflects information not scale), and Solo AUC.

Save CSVs to ``artifacts/results/`` and plots to ``artifacts/plots/``.

In [ ]:
import torch
import torch.nn.functional as F

from src.eval import (
    component_decomposition_table,
    feature_ablation_table,
    plot_component_variance,
    plot_feature_ablation,
)
from src.item_features import POOL_FEATURE_NAMES
from src.models import build_model as _build_model_for_diag
from src.train import evaluate_model as _eval_for_diag


def _predict_with_ablation(
    model: torch.nn.Module,
    val_ds_split,
    *,
    device: str,
    batch_size: int,
    bf16: bool,
    pool_mask: np.ndarray | None = None,
    zero_cluster: bool = False,
    force_alpha_one: bool = False,
    force_beta_zero: bool = False,
    zero_mlp: bool = False,
    judge_mask: np.ndarray | None = None,
    zero_judge: bool = False,
    nn_mask: np.ndarray | None = None,
    zero_nn: bool = False,
) -> tuple[np.ndarray, np.ndarray]:
    """Run val inference with a specific channel zeroed / forced.

    Returns ``(probs, y_true)``. Designed to call the same forward path as
    training: pool features are multiplied element-wise by ``pool_mask``
    (default all-ones) so individual features can be zeroed surgically.
    Judge features behave the same way: ``judge_mask`` is a length-4 mask
    over ``[lp_yes, lp_no, lp_diff, p_yes]`` and ``zero_judge=True`` zeroes
    the whole 4-vector for an all-out without_judge ablation.

    NN features (``[passrate_mean, passrate_weighted_mean, passrate_std,
    coverage, top1_label, top1_similarity, mean_similarity,
    n_labeled_neighbors_log1p]``) get the same treatment: ``nn_mask`` is a
    length-8 mask for per-feature ablation and ``zero_nn=True`` zeroes the
    entire NN vector (the ``without_nn`` ablation).
    """
    model.eval()
    loader = torch.utils.data.DataLoader(
        val_ds_split,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=device.startswith("cuda"),
    )
    autocast = (
        torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=bf16)
        if device.startswith("cuda")
        else torch.amp.autocast("cpu", enabled=False)
    )
    preds, targets = [], []
    with torch.inference_mode():
        with autocast:
            for batch in loader:
                s, bc, ie, se, pf, ci, jf, nn_t, y = [
                    b.to(device, non_blocking=True) for b in batch
                ]
                se_use = se if se.shape[-1] > 0 else None
                pf_use = pf if pf.shape[-1] > 0 else None
                ci_use = ci if ci.numel() > 0 else None
                jf_use = jf if jf.shape[-1] > 0 else None
                nn_use = nn_t if nn_t.shape[-1] > 0 else None
                if pf_use is not None and pool_mask is not None:
                    mask = torch.from_numpy(np.asarray(pool_mask, dtype=np.float32)).to(
                        pf_use.device
                    )
                    pf_use = pf_use * mask
                if zero_cluster and ci_use is not None:
                    ci_use = torch.zeros_like(ci_use)
                if jf_use is not None:
                    if judge_mask is not None:
                        jmask = torch.from_numpy(
                            np.asarray(judge_mask, dtype=np.float32)
                        ).to(jf_use.device)
                        jf_use = jf_use * jmask
                    if zero_judge:
                        jf_use = torch.zeros_like(jf_use)
                if nn_use is not None:
                    if nn_mask is not None:
                        nmask = torch.from_numpy(
                            np.asarray(nn_mask, dtype=np.float32)
                        ).to(nn_use.device)
                        nn_use = nn_use * nmask
                    if zero_nn:
                        nn_use = torch.zeros_like(nn_use)
                kwargs: dict = {}
                if force_alpha_one and getattr(model, "has_irt_heads", False):
                    kwargs["override_alpha"] = torch.ones(
                        s.shape[0], device=device, dtype=torch.float32
                    )
                if force_beta_zero and getattr(model, "has_irt_heads", False):
                    kwargs["override_beta"] = torch.zeros(
                        s.shape[0], device=device, dtype=torch.float32
                    )
                if zero_mlp and getattr(model, "has_residual", False) and hasattr(
                    model, "lambda_resid"
                ):
                    kwargs["override_mlp_zero"] = True
                try:
                    logits = model(
                        s, bc, ie, se_use, pf_use, ci_use, jf_use, nn_use, **kwargs
                    )
                except TypeError:
                    # The model doesn't expose override kwargs (e.g. pure
                    # kfactor); fall back to the plain positional forward.
                    logits = model(s, bc, ie, se_use, pf_use, ci_use, jf_use, nn_use)
                probs = torch.sigmoid(logits).float().cpu().numpy()
                preds.append(probs)
                targets.append(y.float().cpu().numpy())
    return np.concatenate(preds), np.concatenate(targets)


def _decompose_on_val(
    model: torch.nn.Module,
    val_ds_split,
    *,
    device: str,
    batch_size: int,
    bf16: bool,
) -> tuple[dict[str, np.ndarray], np.ndarray]:
    """Run ``model.decompose`` across val_ds_split and stack components."""
    model.eval()
    loader = torch.utils.data.DataLoader(
        val_ds_split,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=device.startswith("cuda"),
    )
    autocast = (
        torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=bf16)
        if device.startswith("cuda")
        else torch.amp.autocast("cpu", enabled=False)
    )
    parts: dict[str, list[np.ndarray]] = {}
    ys: list[np.ndarray] = []
    with torch.inference_mode():
        with autocast:
            for batch in loader:
                s, bc, ie, se, pf, ci, jf, nn_t, y = [
                    b.to(device, non_blocking=True) for b in batch
                ]
                se_use = se if se.shape[-1] > 0 else None
                pf_use = pf if pf.shape[-1] > 0 else None
                ci_use = ci if ci.numel() > 0 else None
                jf_use = jf if jf.shape[-1] > 0 else None
                nn_use = nn_t if nn_t.shape[-1] > 0 else None
                d = model.decompose(
                    s, bc, ie, se_use, pf_use, ci_use, jf_use, nn_use
                )
                for name, tensor in d.items():
                    # We only care about per-row scalar components for the
                    # table (irt / offset / mlp / factor). Skip multi-dim
                    # auxiliary entries like theta/beta_i/alpha_i for the
                    # table but expose them via the dict if useful later.
                    if tensor.dim() == 1:
                        parts.setdefault(name, []).append(
                            tensor.float().cpu().numpy()
                        )
                ys.append(y.float().cpu().numpy())
    stacked = {k: np.concatenate(v) for k, v in parts.items()}
    return stacked, np.concatenate(ys)


PLOTS_DIR = ROOT / CFG["eval"]["plots_dir"]
RESULTS_DIR = (ROOT / CFG["eval"]["results_path"]).parent
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

best_row = (
    runs_df.sort_values("best_val_log_loss", ascending=True).iloc[0]
    if len(runs_df)
    else None
)
if best_row is None:
    print("No trained runs found; skipping diagnostic.")
else:
    BEST_RUN_ID = str(best_row["run_id"])
    BEST_MODEL_NAME = str(best_row["model_name"])
    BEST_CKPT = Path(str(best_row["checkpoint_path"]))
    print(f"Diagnostic on best run: {BEST_RUN_ID}  (model={BEST_MODEL_NAME})")

    ck_diag = torch.load(BEST_CKPT, map_location="cpu")
    mcfg_diag = ModelConfig(**dict(ck_diag["model_cfg"]))
    diag_device = "cuda" if torch.cuda.is_available() else "cpu"
    diag_bs = max(int(CFG["train"]["batch_size"]), 4096)
    diag_bf16 = bool(CFG["encoder"]["bf16"])

    mdl_diag = _build_model_for_diag(BEST_MODEL_NAME, mcfg_diag).to(diag_device)
    mdl_diag.load_state_dict(ck_diag["model_state"], strict=False)

    diag_pf_z = pool_features_z if getattr(mcfg_diag, "use_pool_features", False) else None
    diag_ca = cluster_assignments if getattr(mcfg_diag, "use_cluster_features", False) else None
    diag_jl = (
        judge_features_lookup
        if getattr(mcfg_diag, "use_judge_features", False)
        else None
    )
    diag_nn_train = (
        nn_train_matrix if getattr(mcfg_diag, "use_nn_features", False) else None
    )
    diag_nn_val = (
        nn_val_matrix if getattr(mcfg_diag, "use_nn_features", False) else None
    )
    _, val_ds_diag = _build_arrays(
        primary,
        indexer,
        item_emb_lookup,
        subject_emb_lookup,
        use_subject_emb=USE_SUBJECT_EMB,
        pool_features_z=diag_pf_z,
        cluster_assignments=diag_ca,
        judge_lookup=diag_jl,
        nn_train=diag_nn_train,
        nn_val=diag_nn_val,
    )

    # ----- Analysis A: feature ablation ------------------------------------
    ablations: dict[str, tuple[np.ndarray, np.ndarray]] = {}
    p_full, y_full = _predict_with_ablation(
        mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs, bf16=diag_bf16
    )
    ablations["full"] = (p_full, y_full)

    if getattr(mcfg_diag, "use_pool_features", False):
        mask_zero = np.zeros(int(mcfg_diag.pool_feature_dim), dtype=np.float32)
        ablations["without_pool"] = _predict_with_ablation(
            mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs,
            bf16=diag_bf16, pool_mask=mask_zero,
        )
        # individual pool features
        names = list(POOL_FEATURE_NAMES)
        for i, name in enumerate(names[: int(mcfg_diag.pool_feature_dim)]):
            mask_one = np.ones(int(mcfg_diag.pool_feature_dim), dtype=np.float32)
            mask_one[i] = 0.0
            ablations[f"pool_feature[{name}]"] = _predict_with_ablation(
                mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs,
                bf16=diag_bf16, pool_mask=mask_one,
            )

    if getattr(mcfg_diag, "use_cluster_features", False):
        ablations["without_cluster"] = _predict_with_ablation(
            mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs,
            bf16=diag_bf16, zero_cluster=True,
        )

    if getattr(mdl_diag, "has_irt_heads", False):
        ablations["without_alpha"] = _predict_with_ablation(
            mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs,
            bf16=diag_bf16, force_alpha_one=True,
        )
        ablations["without_beta"] = _predict_with_ablation(
            mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs,
            bf16=diag_bf16, force_beta_zero=True,
        )

    if getattr(mdl_diag, "has_residual", False):
        ablations["without_mlp"] = _predict_with_ablation(
            mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs,
            bf16=diag_bf16, zero_mlp=True,
        )

    # LLM-as-judge ablations: drop the whole 4-vector and each scalar
    # individually so the diagnostic shows whether the head leans on
    # lp_yes / lp_no / lp_diff / p_yes_renorm individually.
    if getattr(mdl_diag, "has_judge_features", False):
        ablations["without_judge"] = _predict_with_ablation(
            mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs,
            bf16=diag_bf16, zero_judge=True,
        )
        judge_feature_names = ("lp_yes", "lp_no", "lp_diff", "p_yes_renorm")
        for i, name in enumerate(judge_feature_names):
            mask = np.ones(int(mcfg_diag.judge_feature_dim), dtype=np.float32)
            mask[i] = 0.0
            ablations[f"judge[{name}]"] = _predict_with_ablation(
                mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs,
                bf16=diag_bf16, judge_mask=mask,
            )

    # NN-feature ablations: drop the whole 8-vector and each scalar
    # individually. This is the headline diagnostic for whether the NN
    # channel is actually contributing -- if without_nn shows a large
    # Delta NLL, the model has learned to lean on the neighbor signal.
    # The per-feature breakdown shows whether the residual MLP is using
    # passrate_mean / weighted_mean / coverage etc. as expected.
    if getattr(mdl_diag, "has_nn_features", False):
        ablations["without_nn"] = _predict_with_ablation(
            mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs,
            bf16=diag_bf16, zero_nn=True,
        )
        nn_feature_names_local = (
            "passrate_mean",
            "passrate_weighted_mean",
            "passrate_std",
            "coverage",
            "top1_label",
            "top1_similarity",
            "mean_similarity",
            "n_labeled_neighbors_log1p",
        )
        nn_dim_local = int(mcfg_diag.nn_feature_dim)
        for i, name in enumerate(nn_feature_names_local[:nn_dim_local]):
            mask = np.ones(nn_dim_local, dtype=np.float32)
            mask[i] = 0.0
            ablations[f"nn[{name}]"] = _predict_with_ablation(
                mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs,
                bf16=diag_bf16, nn_mask=mask,
            )

    ablation_df = feature_ablation_table(ablations, full_key="full")
    ablation_csv = RESULTS_DIR / f"feature_ablation_{BEST_RUN_ID}.csv"
    ablation_df.to_csv(ablation_csv, index=False)
    ablation_plot = PLOTS_DIR / f"feature_ablation_{BEST_RUN_ID}.png"
    plot_feature_ablation(
        ablation_df, ablation_plot, title=f"Feature ablation: {BEST_RUN_ID}"
    )
    print("\n=== Analysis A: feature ablation ===")
    print(ablation_df.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    print(f"Wrote {ablation_csv.relative_to(ROOT)}")
    print(f"Wrote {ablation_plot.relative_to(ROOT)}")

    # ----- Analysis B: logit-component decomposition ----------------------
    components, y_dec = _decompose_on_val(
        mdl_diag, val_ds_diag, device=diag_device, batch_size=diag_bs, bf16=diag_bf16
    )
    decomp_df = component_decomposition_table(components, y_dec)
    decomp_csv = RESULTS_DIR / f"component_decomp_{BEST_RUN_ID}.csv"
    decomp_df.to_csv(decomp_csv, index=False)
    decomp_plot = PLOTS_DIR / f"component_decomp_{BEST_RUN_ID}.png"
    plot_component_variance(
        components, decomp_plot, title=f"Component variance: {BEST_RUN_ID}"
    )
    print("\n=== Analysis B: logit-component decomposition ===")
    print(decomp_df.to_string(index=False, float_format=lambda v: f"{v:.5f}"))
    print(f"Wrote {decomp_csv.relative_to(ROOT)}")
    print(f"Wrote {decomp_plot.relative_to(ROOT)}")

    # One-line summaries
    non_full = ablation_df[ablation_df["channel_removed"] != "full"]
    if not non_full.empty:
        top = non_full.sort_values("delta_nll", ascending=False).iloc[0]
        print(
            f"\nLargest single contributor: {top['channel_removed']} "
            f"(Δ NLL = {float(top['delta_nll']):.5f})"
        )
    if not decomp_df.empty:
        best_comp = decomp_df.sort_values("solo_nll", ascending=True).iloc[0]
        print(
            f"Best solo component: {best_comp['component']} "
            f"(Solo NLL = {float(best_comp['solo_nll']):.5f})"
        )

## 15. Random-row overfit flag

If a residual model only improves random-row val but not item-cold-start
val, the residual is overfitting. We print a warning when this happens.

In [ ]:
def _mean_metric(df, model, split, metric="val_log_loss"):
    sub = df[(df["model_name"] == model) & (df["split"] == split)]
    return float(sub[metric].mean()) if len(sub) else float("nan")


for residual_name in (
    "kfactor_mlp",
    "kfactor_gated_mlp",
    "kfactor_irt_item",
    "kfactor_irt_item_mlp",
    "kfactor_irt_item_gated_mlp",
):
    base = "kfactor"
    if "random_row_debug" not in {row["split"] for row in all_results}:
        continue
    base_ic = _mean_metric(results_df, base, "item_cold_start")
    res_ic = _mean_metric(results_df, residual_name, "item_cold_start")
    base_rr = _mean_metric(results_df, base, "random_row_debug")
    res_rr = _mean_metric(results_df, residual_name, "random_row_debug")
    improves_random = res_rr < base_rr - 1e-4
    improves_cold = res_ic < base_ic - 1e-4
    if improves_random and not improves_cold:
        print(
            f"WARN: {residual_name} improves random-row "
            f"({base_rr:.5f} -> {res_rr:.5f}) but NOT item-cold-start "
            f"({base_ic:.5f} -> {res_ic:.5f}). Likely overfitting."
        )
    else:
        print(
            f"OK  : {residual_name} item-cold-start {base_ic:.5f} -> {res_ic:.5f} "
            f"(random-row {base_rr:.5f} -> {res_rr:.5f})"
        )

## 16. Slicewise metrics: by benchmark / condition / subject family / token length

In [ ]:
attached = attach_subject_family(primary.val.copy())

# Pick the BEST run for slicewise plots
best_run = (
    runs_df.sort_values("best_val_log_loss", ascending=True).iloc[0]
    if len(runs_df)
    else None
)
if best_run is not None:
    p_best, y_best = _predict_with_checkpoint(
        best_run["model_name"], Path(best_run["checkpoint_path"]), primary
    )
    attached["_pred"] = p_best
    by_bench = metrics_by_group(attached, group_col="benchmark", min_n=20)
    by_cond = metrics_by_group(attached, group_col="condition", min_n=20)
    by_fam = metrics_by_group(attached, group_col="subject_family", min_n=50)
    print("=== by benchmark ===")
    print(by_bench.to_string(index=False))
    print("=== by condition ===")
    print(by_cond.to_string(index=False))
    print("=== by subject family ===")
    print(by_fam.to_string(index=False))

## 17. Plots: log-loss by model, calibration curves, by benchmark, residual delta

In [ ]:
from src.eval import (
    plot_calibration_curves,
    plot_logloss_by_benchmark,
    plot_residual_improvement_by_benchmark,
    plot_val_logloss_by_model,
)

PLOTS = ROOT / CFG["eval"]["plots_dir"]
PLOTS.mkdir(parents=True, exist_ok=True)

primary_results = results_df[results_df["split"] == "item_cold_start"]
plot_val_logloss_by_model(primary_results, PLOTS / "val_logloss_by_model.png")

per_run_cal: dict[str, tuple[np.ndarray, np.ndarray]] = {}
for r in ALL_RUNS:
    rid = r["run_id"]
    if rid in per_run_split_predictions.get("item_cold_start", {}):
        per_run_cal[rid] = per_run_split_predictions["item_cold_start"][rid]
if per_run_cal:
    plot_calibration_curves(per_run_cal, PLOTS / "calibration_curves.png")

per_run_per_bench = {}
for r in ALL_RUNS:
    p_val, y_val = per_run_split_predictions.get("item_cold_start", {}).get(
        r["run_id"], (None, None)
    )
    if p_val is None:
        continue
    val = primary.val.copy()
    val["_pred"] = p_val
    per_run_per_bench[r["run_id"]] = metrics_by_group(val, group_col="benchmark", min_n=20)
plot_logloss_by_benchmark(per_run_per_bench, PLOTS / "logloss_by_benchmark.png")

# Residual deltas: kfactor vs kfactor_mlp / kfactor_gated_mlp (averaged over seeds)
def _avg_per_bench(model_name: str):
    parts = []
    for r in ALL_RUNS:
        if r["model_name"] != model_name:
            continue
        p_val, y_val = per_run_split_predictions.get("item_cold_start", {}).get(
            r["run_id"], (None, None)
        )
        if p_val is None:
            continue
        v = primary.val.copy()
        v["_pred"] = p_val
        parts.append(metrics_by_group(v, group_col="benchmark", min_n=20))
    if not parts:
        return pd.DataFrame(columns=["benchmark", "log_loss"])
    return (
        pd.concat(parts)
        .groupby("benchmark")["log_loss"]
        .mean()
        .reset_index()
    )


base_pb = _avg_per_bench("kfactor")
for chal in (
    "kfactor_mlp",
    "kfactor_gated_mlp",
    "kfactor_irt_item",
    "kfactor_irt_item_mlp",
    "kfactor_irt_item_gated_mlp",
):
    chal_pb = _avg_per_bench(chal)
    if not base_pb.empty and not chal_pb.empty:
        plot_residual_improvement_by_benchmark(
            base_pb,
            chal_pb,
            PLOTS / f"residual_delta_{chal}_vs_kfactor.png",
            base_label="kfactor",
            challenger_label=chal,
        )

# Performance vs item token length
import collections

token_len_map: dict[str, int] = {}
if not bool(CFG["encoder"].get("use_random_embeddings", False)):
    # The encoder's stats record per-batch lengths; we don't have a
    # per-item map directly. Fall back to character length / 4 as a cheap
    # proxy for the plot.
    pass
primary_val = primary.val.copy()
primary_val["item_token_len"] = (
    primary_val["item_content"].astype(str).str.len() // 4 + 1
)
per_run_by_len = {}
for r in ALL_RUNS:
    p_val, y_val = per_run_split_predictions.get("item_cold_start", {}).get(
        r["run_id"], (None, None)
    )
    if p_val is None:
        continue
    v = primary_val.copy()
    v["_pred"] = p_val
    per_run_by_len[r["run_id"]] = metrics_by_token_length(v)
from src.eval import plot_perf_vs_token_length
plot_perf_vs_token_length(per_run_by_len, PLOTS / "perf_vs_item_length.png")

print(f"Plots written to {PLOTS.resolve()}")

## 18. Choose a trained run to export

Default: best item-cold-start val log-loss. Override by setting
``SELECTED_RUN_ID`` below. If ``ipywidgets`` is installed, a dropdown lets
you pick interactively.

In [ ]:
SELECTED_RUN_ID: str | None = None  # set manually to override

try:
    import ipywidgets as widgets  # type: ignore
    from IPython.display import display  # type: ignore

    options = [
        (
            f"{r['run_id']}  ll={r['best_val_log_loss']:.5f}",
            r["run_id"],
        )
        for _, r in runs_df.iterrows()
    ]
    dropdown = widgets.Dropdown(options=options, description="Run:")
    display(dropdown)
    selected_widget = dropdown
except Exception:
    selected_widget = None

if SELECTED_RUN_ID is None:
    if selected_widget is not None and selected_widget.value:
        SELECTED_RUN_ID = selected_widget.value
    elif len(runs_df) > 0:
        SELECTED_RUN_ID = runs_df.iloc[0]["run_id"]
print(f"SELECTED_RUN_ID = {SELECTED_RUN_ID}")

## 19. Export the selected run as a submission folder + zip

Produces:
- ``submission/model.py`` (self-contained runtime)
- ``submission/labeling.py`` (uncertainty acquisition)
- ``submission/models.txt``
- ``submission/requirements.txt``
- ``submission/artifacts/checkpoint.pt`` + ``runtime_meta.json``
- ``submission.zip``

In [ ]:
from src.export_submission import bundle_training_cache, export_run, make_submission_zip
from src.train import TrainResult

selected = runs_df[runs_df["run_id"] == SELECTED_RUN_ID].iloc[0]
selected_dict = selected.to_dict()
selected_meta = json.loads(Path(selected_dict["metadata_path"]).read_text())
sel_result = TrainResult(
    run_id=selected_dict["run_id"],
    model_name=selected_dict["model_name"],
    seed=int(selected_dict["seed"]),
    k=int(selected_dict["k"]),
    epoch_best=int(selected_meta["result"]["epoch_best"]),
    best_val_log_loss=float(selected_dict["best_val_log_loss"]),
    best_val_brier=float(selected_dict["best_val_brier"]),
    best_val_auc=(
        float(selected_dict["best_val_auc"])
        if selected_dict["best_val_auc"] is not None
        else None
    ),
    history=list(selected_meta["result"]["history"]),
    checkpoint_path=str(selected_dict["checkpoint_path"]),
    metadata_path=str(selected_dict["metadata_path"]),
    n_train=int(selected_meta["result"].get("n_train", 0)),
    n_val=int(selected_meta["result"].get("n_val", 0)),
    elapsed_seconds=float(selected_dict.get("elapsed_seconds", 0.0)),
)

# 19a. The *selected checkpoint's* model_cfg is the single source of truth
# for what the runtime needs to ship: ambient notebook flags
# (USE_NN_FEATURES, USE_JUDGE_FEATURES) reflect this run's *config*, but if
# the user toggled them off between training and export the trained head
# would silently see all-zero feature vectors at inference. We re-read the
# checkpoint here and let it override the notebook-level flags.
import torch as _torch_export_peek
_ckpt_peek = _torch_export_peek.load(
    sel_result.checkpoint_path, map_location="cpu", weights_only=False
)
_ckpt_model_cfg = dict(_ckpt_peek.get("model_cfg") or {})
del _ckpt_peek  # release before the cache build allocates GBs
CKPT_USE_NN_FEATURES = bool(_ckpt_model_cfg.get("use_nn_features", False))
CKPT_USE_JUDGE_FEATURES = bool(_ckpt_model_cfg.get("use_judge_features", False))
print(
    f"Checkpoint model_cfg: use_nn_features={CKPT_USE_NN_FEATURES} "
    f"use_judge_features={CKPT_USE_JUDGE_FEATURES} "
    f"(notebook flags: USE_NN_FEATURES={USE_NN_FEATURES} "
    f"USE_JUDGE_FEATURES={USE_JUDGE_FEATURES})"
)
if CKPT_USE_NN_FEATURES and not USE_NN_FEATURES:
    raise RuntimeError(
        "Selected checkpoint was trained with use_nn_features=True but the "
        "current notebook session has USE_NN_FEATURES=False (so cell 8d did "
        "not build the NN passrate table). Re-run cell 8d with CFG.nn_features"
        ".enabled=True before exporting, or pick a checkpoint trained without "
        "NN features."
    )
if CKPT_USE_JUDGE_FEATURES and not USE_JUDGE_FEATURES:
    raise RuntimeError(
        "Selected checkpoint was trained with use_judge_features=True but the "
        "current notebook session has USE_JUDGE_FEATURES=False. Re-run with "
        "CFG.judge.enabled=True before exporting, or pick a non-judge checkpoint."
    )

# 19b. Build the quantized training-item cache (int8 + optional PCA + FAISS).
# This is the artifact shipped inside submission/cache/ for runtime nearest-
# neighbor lookup. Fails loudly if max_bundle_size_mb is exceeded.
training_cache_dir = ROOT / "artifacts" / "submission_cache"
submission_cache_cfg = CFG.get("submission_cache", {}) or {}
training_cache_result = None

# Resolve the NN feature config + subject_to_id mapping built in cell 8d.
# Both are needed by bundle_training_cache so the runtime can compute the
# same 8-scalar NN feature vector the model was trained on -- without
# them, an NN-aware model loads the cache but gets all zeros for the NN
# channel and degrades silently. We gate on the *checkpoint's* verdict.
#
# We deliberately use the *indexer*'s subject_to_id rather than
# `nn_subject_to_id_map` built in cell 8d. Both are constructed by the same
# deterministic "first-seen" algorithm over `primary.train["subject_key"]`,
# but the indexer is the mapping the trained model and its checkpoint
# actually use, so the runtime's `_SUBJECT_TO_ID` and the NN passrate row
# index are guaranteed to agree.
nn_cfg_for_export = NN_FEATURES_CFG_DICT if CKPT_USE_NN_FEATURES else None
subject_to_id_for_export = (
    dict(indexer.subject_to_id) if CKPT_USE_NN_FEATURES else None
)
if CKPT_USE_NN_FEATURES and nn_subject_to_id_map:
    drift = [
        k for k in nn_subject_to_id_map
        if nn_subject_to_id_map[k] != indexer.subject_to_id.get(k, -1)
    ]
    if drift:
        raise RuntimeError(
            "NN subject_to_id (cell 8d) and Indexer.subject_to_id (cell 10) "
            f"disagree on {len(drift)} keys; cannot ship NN features safely. "
            f"First diverging keys: {drift[:5]}"
        )

if bool(submission_cache_cfg.get("enabled", True)):
    cluster_assign_map = (
        dict(cluster_assignments) if cluster_assignments is not None else None
    )
    training_cache_result = bundle_training_cache(
        items_parquet_path=embedder.items_path,
        out_dir=training_cache_dir,
        submission_cache_cfg=submission_cache_cfg,
        encoder_cfg=CFG["encoder"],
        items_meta_df=item_df,
        cluster_assignments=cluster_assign_map,
        n_clusters=N_CLUSTERS if USE_CLUSTER_FEATURES else 0,
        train_df=primary.train,
        nn_features_cfg=nn_cfg_for_export,
        subject_to_id=subject_to_id_for_export,
    )
    print(
        f"Training cache: {training_cache_result.total_mb:.2f} MB at "
        f"{training_cache_dir.relative_to(ROOT)}"
    )
    for fname, mb in training_cache_result.sizes_mb.items():
        print(f"  {fname:32s} {mb:7.2f} MB")
    print(
        f"NN features in cache: enabled={CKPT_USE_NN_FEATURES} "
        f"n_subjects_indexed={len(subject_to_id_for_export or {})} "
        f"runtime_k={int(submission_cache_cfg.get('runtime_k', nn_cfg.k if USE_NN_FEATURES else 0))}"
    )
else:
    print("submission_cache.enabled = false; not shipping training-item cache")

# 19c. Materialize the submission directory. export_run also enforces the
# checkpoint <-> caller-config consistency described above and will raise
# if e.g. the NN cache files are missing while the checkpoint expects them.
sub_dir = export_run(
    result=sel_result,
    encoder_cfg=CFG["encoder"],
    submission_dir=ROOT / CFG["submission"]["dir"],
    include_labeling=True,
    extra_models_txt=None,
    pool_stats_path=POOL_STATS_PATH if USE_POOL_FEATURES else None,
    cluster_centroids_path=CENTROIDS_PATH if USE_CLUSTER_FEATURES else None,
    pool_feature_names=list(POOL_FEATURE_NAMES),
    training_cache_dir=training_cache_dir if training_cache_result is not None else None,
    judge_cfg=CFG.get("judge") if CKPT_USE_JUDGE_FEATURES else None,
    nn_features_cfg=nn_cfg_for_export,
)
print(
    f"Judge in bundle     : ckpt_use_judge={CKPT_USE_JUDGE_FEATURES} "
    f"ship_at_runtime={bool((CFG.get('judge') or {}).get('ship_at_runtime', True))} "
    f"model_id={(CFG.get('judge') or {}).get('model_id', '')}"
)

# 19d. Zip the submission folder. We enforce a final-zip size cap (default
# 70 MB) -- anything bigger has historically failed the Codabench upload
# widget, and silently shipping an un-uploadable bundle is worse than
# failing here.
max_zip_size_mb = float(
    (CFG.get("submission") or {}).get("max_zip_size_mb", 70)
)
zip_path = make_submission_zip(
    submission_dir=sub_dir,
    zip_path=ROOT / CFG["submission"]["zip_path"],
    max_zip_size_mb=max_zip_size_mb,
)
sub_bundle_mb = sum(
    p.stat().st_size for p in sub_dir.rglob("*") if p.is_file()
) / (1024 * 1024)
print(f"Submission ready: {sub_dir}")
print(f"Submission size : {sub_bundle_mb:.2f} MB (uncompressed)")
print(f"Zip             : {zip_path}  ({zip_path.stat().st_size / (1024*1024):.2f} MB)")
print(f"Zip cap         : {max_zip_size_mb:.0f} MB")

## 20. Submission smoke test (notebook variant)

Imports ``submission/model.py``, calls predict() on 20 held-out rows,
checks output types, range, finiteness, and timing.

In [ ]:
import importlib
import io

sub_dir_str = str(sub_dir.resolve())
if sub_dir_str not in sys.path:
    sys.path.insert(0, sub_dir_str)
if "model" in sys.modules:
    del sys.modules["model"]
sub_model = importlib.import_module("model")

# Sanity check: an NN-aware checkpoint must ship the matching NN cache.
# Without it the runtime returns all zeros for the 8 NN scalars and the
# bundle silently regresses to a different model than the one we trained.
_runtime_meta = sub_model.META
_nn_meta = (_runtime_meta.get("nn_features") or {})
_use_nn_runtime = bool(getattr(sub_model, "USE_NN_FEATURES", False))
_nn_cache_ok = bool(
    getattr(sub_model, "TRAINING_CACHE", None) is not None
    and getattr(sub_model.TRAINING_CACHE, "nn_passrate", None) is not None
)
if bool(_nn_meta.get("enabled", False)) and not _nn_cache_ok:
    raise RuntimeError(
        "Bundle declares NN features in runtime_meta.json but the training "
        "cache does not contain the passrate matrix. Re-run cell 19 with "
        "nn_features_cfg + subject_to_id wired through bundle_training_cache."
    )
print(
    f"Runtime NN features : enabled={bool(_nn_meta.get('enabled'))} "
    f"runtime_k={int(_nn_meta.get('runtime_k', 0))} "
    f"feature_dim={int(_nn_meta.get('feature_dim', 0))} "
    f"cache_ok={_nn_cache_ok}"
)

smoke_rows = primary.val.sample(n=min(20, len(primary.val)), random_state=0)
ok = True
t0 = time.time()
for _, row in smoke_rows.iterrows():
    inp = {
        "benchmark": str(row["benchmark"]),
        "condition": str(row["condition"]),
        "subject_content": str(row["subject_content"]),
        "item_content": str(row["item_content"]),
    }
    p = sub_model.predict(inp, None)
    if not (isinstance(p, float) and np.isfinite(p) and 0.0 <= p <= 1.0):
        print(f"FAIL: {p!r} for inp keys {list(inp)}")
        ok = False
print(f"Smoke test (predict): {'OK' if ok else 'FAIL'} -- elapsed {time.time() - t0:.2f}s")

# 20b. Nearest-neighbor lookup smoke test. Pick 3 val items, fetch the
# encoder embedding from the model module, and verify TRAINING_CACHE
# returns well-formed (indices, scores) of the right shape and value range.
training_cache = getattr(sub_model, "TRAINING_CACHE", None)
nn_ok = True
if training_cache is None:
    print("NN smoke test: SKIP (TRAINING_CACHE not loaded)")
else:
    K_NN = 10
    nn_rows = primary.val.sample(n=min(3, len(primary.val)), random_state=1)
    n_total = int(training_cache.embeddings_q.shape[0])
    for _, row in nn_rows.iterrows():
        item_emb = sub_model._get_item_embedding(
            str(row["benchmark"]),
            str(row["condition"]),
            str(row["item_content"]),
        )
        idx, scores = training_cache.nearest(item_emb, k=K_NN)
        if idx.shape != (K_NN,) or scores.shape != (K_NN,):
            print(f"FAIL: bad NN shapes idx={idx.shape} scores={scores.shape}")
            nn_ok = False
            continue
        if not np.all(np.isfinite(scores)):
            print(f"FAIL: non-finite NN scores {scores}")
            nn_ok = False
            continue
        if int(idx.min()) < 0 or int(idx.max()) >= n_total:
            print(f"FAIL: NN indices out of range: min={idx.min()} max={idx.max()} n={n_total}")
            nn_ok = False
            continue
        print(
            f"NN OK item_key={str(row['item_key'])[:8]}... "
            f"top-{K_NN} score range [{float(scores.min()):.3f}, {float(scores.max()):.3f}]"
        )
    print(f"Smoke test (NN lookup): {'OK' if nn_ok else 'FAIL'}")

## 21. Optional GCS sync

If you set ``CFG['gcs']['bucket']`` to a `gs://...` prefix, this cell copies
the ``artifacts/`` and ``submission/`` directories there. Never syncs
environment variables, tokens, or .ipynb_checkpoints.

In [ ]:
gcs_bucket = (CFG.get("gcs") or {}).get("bucket")
if gcs_bucket and CFG.get("gcs", {}).get("sync_artifacts", False):
    from google.cloud import storage  # type: ignore

    client = storage.Client()
    bucket_name = gcs_bucket.replace("gs://", "").split("/")[0]
    prefix = "/".join(gcs_bucket.replace("gs://", "").split("/")[1:]).strip("/")
    bucket = client.bucket(bucket_name)
    for root_dir in ("artifacts", "submission"):
        rd = ROOT / root_dir
        if not rd.exists():
            continue
        for path in rd.rglob("*"):
            if not path.is_file():
                continue
            rel = path.relative_to(ROOT)
            key = f"{prefix}/{rel.as_posix()}" if prefix else rel.as_posix()
            blob = bucket.blob(key)
            blob.upload_from_filename(str(path))
    print(f"Synced artifacts/ and submission/ to {gcs_bucket}")
else:
    print("GCS sync skipped (CFG['gcs']['bucket'] not set or sync_artifacts=False)")